# Modelo SARIMAX - PBI Electricidad y Agua con selección automática de variables exógenas (BCRP / IGP)
## Versión para Kaggle con configuración ampliada

### Docente: Dr. Richard Fernández

## 0. Ejecución en Kaggle

1. En Kaggle: **Create → New Notebook → File → Import Notebook** y sube este archivo.
2. En el panel **Settings** (derecha):
   - **Accelerator: GPU T4 x2.** La P100 no es compatible con las versiones recientes de PyTorch; si se elige, la celda siguiente lo detecta y las redes se entrenan en CPU.
   - **Internet: on.** Lo requieren la API del BCRP, el ICEN del IGP, la instalación de paquetes y la descarga de los modelos de Hugging Face. Kaggle pide verificar la cuenta con un teléfono para activarlo.
3. Si la API del BCRP no responde desde Kaggle, sube `datos_bcrp_igp.json` como Dataset (**Add Input → Upload**). El notebook lo busca en `/kaggle/input` y lo usa en lugar de la descarga.
4. Ejecuta con **Save Version → Save & Run All (Commit)**: la ejecución continúa aunque se cierre el navegador. El límite de Kaggle es de 12 horas por ejecución.
5. Al terminar, las figuras están en `/kaggle/working/Image` y las tablas, el modelo y los datos descargados en `/kaggle/working` (pestaña **Output** de la versión guardada).

| Etapa | Notebook original | Versión Kaggle |
|---|---|---|
| SARIMA: órdenes p, q / P, Q | 0–3 / 0–1 | 0–4 / 0–2 |
| ARIMA sin parte estacional: p, q | 0–3 | 0–5 |
| Rezago máximo de las exógenas | 12 meses | 24 meses |
| Prophet: combinaciones de hiperparámetros | 4 × 3 = 12 | 6 × 4 = 24 |
| Boosting: árboles × profundidad × tasa | 5 × 4 × 2 = 40 | 8 × 6 × 3 = 144, en paralelo |
| H2O AutoML | 10 modelos, 600 s | 40 modelos, 3 600 s |
| Isolation Forest | 1 000 árboles, 10 semillas | 2 000 árboles, 20 semillas |
| Redes recurrentes: épocas | 5 fijas | hasta 50, con parada temprana (paciencia 10; validación con los últimos 36 meses del train) y reentrenamiento con todo el train |
| Redes recurrentes: neuronas (LSTM / GRU / 2.ª capa / densa) | 128 / 50 / 64 / 25 | 128 / 128 / 96 / 32 |
| Redes recurrentes: semillas / validación de variables | 3 / no | 5 / sí |
| Transformer: dimensión / cabezas / capa FF / bloques | 32 / 4 / 64 / 2 | 64 / 8 / 128 / 3 |
| Transformer: largos de parche | 1, 3, 6 | 1, 2, 3, 4, 6 |
| Transformer: épocas máximas / paciencia / semillas | 200 / 25 / 3 | 400 / 40 / 5 |
| Toto 2.0 | 22M, 313M, 1B | 4M, 22M, 313M, 1B, 2.5B |

Todos los valores están en el diccionario `CONFIG`. Los textos del notebook que citan cifras describen la ejecución original; con esta configuración los resultados cambian.


In [ ]:
import glob
import importlib.util
import os
import re
import shutil
import subprocess
import sys

EN_KAGGLE = os.path.isdir('/kaggle/working')
if EN_KAGGLE:
    os.chdir('/kaggle/working')
print('Kaggle:', 'sí' if EN_KAGGLE else 'no', '| carpeta de trabajo:', os.getcwd(), '| Python', sys.version.split()[0])

try:
    import requests
    requests.get('https://pypi.org/simple/pip/', timeout=15).raise_for_status()
    print('Internet: sí')
except Exception as error:
    print(f'Internet: NO ({type(error).__name__}). Activa Settings → Internet on, o sube datos_bcrp_igp.json como Dataset.')

paquetes = {'statsmodels': 'statsmodels', 'xgboost': 'xgboost', 'lightgbm': 'lightgbm', 'catboost': 'catboost',
            'prophet': 'prophet', 'holidays': 'holidays', 'h2o': 'h2o', 'keras': 'keras', 'joblib': 'joblib'}
faltantes = [paquete for modulo, paquete in paquetes.items() if importlib.util.find_spec(modulo) is None]
if faltantes:
    print('Instalando:', ', '.join(faltantes))
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *faltantes])

def version_java(java='java'):
    if shutil.which(java) is None:
        return None
    numero = re.search(r'version "(\d+)(?:\.(\d+))?', subprocess.run([java, '-version'], capture_output=True, text=True).stderr)
    if numero is None:
        return None
    return int(numero.group(2)) if numero.group(1) == '1' else int(numero.group(1))


import h2o

java_sistema = version_java()
if EN_KAGGLE and (java_sistema is None or java_sistema > 17):
    print(f'Java del sistema: {java_sistema} | H2O {h2o.__version__} admite Java 8 a 17 → se instala Java 17 para H2O')
    subprocess.run('apt-get update -qq && apt-get install -y -qq openjdk-17-jre-headless > /dev/null', shell=True, check=True)
    os.environ['H2O_JAVA_HOME'] = sorted(glob.glob('/usr/lib/jvm/java-17-openjdk*'))[0]
java_h2o = os.path.join(os.environ['H2O_JAVA_HOME'], 'bin', 'java') if 'H2O_JAVA_HOME' in os.environ else 'java'
print(f'Java que usará H2O {h2o.__version__}:', version_java(java_h2o))

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
                     capture_output=True, text=True) if shutil.which('nvidia-smi') else None
if gpu is not None and gpu.returncode == 0 and gpu.stdout.strip():
    nombre_gpu, capacidad = [s.strip() for s in gpu.stdout.strip().splitlines()[0].split(',')]
    mayor, menor = map(int, capacidad.split('.'))
    lista = subprocess.run([sys.executable, '-c', 'import torch; print(*torch.cuda.get_arch_list())'],
                           capture_output=True, text=True).stdout.split()
    compilado = [tuple(map(int, m.groups())) for m in (re.fullmatch(r'sm_(\d+)(\d)', a) for a in lista) if m]
    if compilado and not any(m == mayor and n <= menor for m, n in compilado):
        os.environ['CUDA_VISIBLE_DEVICES'] = ''
        print(f'GPU: {nombre_gpu} (sm_{mayor}{menor}) no es compatible con este PyTorch → se usará CPU. '
              'Elige "GPU T4 x2" en Settings → Accelerator.')
    else:
        print(f'GPU: {nombre_gpu} (sm_{mayor}{menor})')
else:
    print('GPU: no detectada → las redes y los Transformers se entrenarán en CPU.')


### Configuración ampliada

In [ ]:
CONFIG = {
    'sarima_pq_max': 4,
    'sarima_PQ_max': 2,
    'sarima_pq_suma_max': 5,
    'arima_pq_max': 5,
    'rezago_max_exog': 24,
    'prophet_cps': (0.001, 0.005, 0.01, 0.05, 0.1, 0.5),
    'prophet_sps': (0.01, 0.1, 1.0, 10.0),
    'boost_folds': 5,
    'boost_arboles': [100, 200, 300, 400, 500, 600, 800, 1000],
    'boost_profundidad': [2, 3, 4, 5, 6, 7],
    'boost_tasa': [0.1, 0.05, 0.01],
    'automl_max_modelos': 40,
    'automl_segundos': 3600,
    'if_arboles': 2000,
    'if_semillas': 20,
    'rnn_epocas_max': 50,
    'rnn_paciencia': 10,
    'rnn_lote': 1,
    'rnn_n_val': 36,
    'rnn_semillas': 5,
    'rnn_validar': True,
    'rnn_neuronas_lstm': 128,
    'rnn_neuronas_gru': 128,
    'rnn_neuronas_capa2': 96,
    'rnn_densa': 32,
    'tr_d_modelo': 64,
    'tr_cabezas': 8,
    'tr_d_ff': 128,
    'tr_bloques': 3,
    'tr_dropout': 0.1,
    'tr_largos_parche': [1, 2, 3, 4, 6],
    'tr_lote': 16,
    'tr_epocas_max': 400,
    'tr_paciencia': 40,
    'tr_n_val': 36,
    'tr_semillas': 5,
    'toto_tamanos': ['4m', '22m', '313m', '1B', '2.5B'],
}


## 1. Librerías

In [ ]:
import gc
import glob
import importlib.util
import io
import json
import logging
import os
import subprocess
import sys
import time
import warnings

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss, acf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.graphics.gofplots import qqplot
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.outliers_influence import variance_inflation_factor
from dateutil.easter import easter
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

os.environ['LOKY_MAX_CPU_COUNT'] = str(os.cpu_count() or 1)

alpha = 0.05


def instalar(modulo, *args_pip):
    if importlib.util.find_spec(modulo) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args_pip])
        importlib.invalidate_caches()

## 2. Llamado a los datasets

In [ ]:
MESES = {'Ene': 1, 'Feb': 2, 'Mar': 3, 'Abr': 4, 'May': 5, 'Jun': 6, 'Jul': 7,
         'Ago': 8, 'Sep': 9, 'Set': 9, 'Oct': 10, 'Nov': 11, 'Dic': 12}
FIN = '2026-07-01'

ARCHIVO_DATOS = 'datos_bcrp_igp.json'
rutas_datos = glob.glob(f'/kaggle/input/**/{ARCHIVO_DATOS}', recursive=True) + glob.glob(ARCHIVO_DATOS)
datos_web = json.load(open(rutas_datos[0], encoding='utf-8')) if rutas_datos else {}
print('Origen de los datos:', rutas_datos[0] if rutas_datos else 'API del BCRP y del IGP')


def leer_json(url):
    if url not in datos_web:
        try:
            datos_web[url] = requests.get(url, timeout=60).json()
        except ValueError:
            datos_web[url] = json.loads(subprocess.run(['curl', '-s', '-m', '60', url], capture_output=True, check=True).stdout)
    return datos_web[url]

def descargar_bcrp(codigo, inicio='1994-1', fin='2026-7'):
    datos = leer_json(f'https://estadisticas.bcrp.gob.pe/estadisticas/series/api/{codigo}/json/{inicio}/{fin}')
    fechas = [pd.Timestamp(int(p['name'][-4:]), MESES[p['name'][:3]], 1) for p in datos['periods']]
    valores = pd.to_numeric([p['values'][0] for p in datos['periods']], errors='coerce')
    return pd.Series(valores, index=pd.DatetimeIndex(fechas), name=datos['config']['series'][0]['name']).asfreq('MS')

def descargar_icen(url='http://met.igp.gob.pe/datos/ICEN.txt'):
    if url not in datos_web:
        datos_web[url] = requests.get(url, timeout=60).text
    texto = datos_web[url]
    tabla = pd.read_csv(io.StringIO(texto), comment='%', sep=r'\s+', header=None, names=['anio', 'mes', 'icen'])
    fechas = pd.to_datetime(dict(year=tabla['anio'], month=tabla['mes'], day=1))
    return pd.Series(tabla['icen'].values, index=fechas, name='ICEN (°C)').asfreq('MS')[:FIN]

indice = descargar_bcrp('PN01765AM')
variacion = descargar_bcrp('PN01723AM', inicio='1995-1')
cobre = descargar_bcrp('PN01652XM', inicio='1990-1')
icen = descargar_icen()

fichas_exogenas = pd.DataFrame([
    ['mineria', 'Minería metálica', 'minería', 'BCRP PN01760AM', 'actividad', '100·ln', 1, '+',
     'Norgate y Haque (2010); BCRP, Moneda 198'],
    ['manuf_primaria', 'Manufactura de recursos primarios', 'manuf. primaria', 'BCRP PN01763AM', 'actividad', '100·ln', 1, '+',
     'Peralta Neyra (2021)'],
    ['manuf_no_primaria', 'Manufactura no primaria', 'manuf. no primaria', 'BCRP PN01764AM', 'actividad', '100·ln', 1, '+',
     'Peralta Neyra (2021)'],
    ['construccion', 'Construcción', 'construcción', 'BCRP PN01766AM', 'actividad', '100·ln', 1, '+', 'Peralta Neyra (2021)'],
    ['cobre', 'Precio del cobre', 'cobre', 'BCRP PN01652XM', 'precio', '100·ln', 1, '+', 'Canal minero (BCRP, Moneda 198)'],
    ['icen', 'ICEN', 'ICEN', 'IGP / ENFEN', 'clima', 'Ninguna', 2, '±', 'Takahashi et al. (2014); Lavado-Casimiro et al. (2013)'],
    ['dias_habiles', 'Días hábiles', 'días hábiles', 'Calendario y feriados nacionales', 'calendario', 'Ninguna', 0, '+',
     'Bell y Hillmer (1983); Hong y Fan (2016)'],
    ['semana_santa', 'Semana Santa', 'Semana Santa', 'Calendario (8 días antes de Pascua)', 'calendario', 'Ninguna', 0, '−',
     'Bell y Hillmer (1983)'],
], columns=['clave', 'Variable', 'Corto', 'Fuente', 'tipo', 'Transformación', 'Rezago mínimo', 'Signo esperado',
            'Sustento']).set_index('clave')

codigos_sectores = {clave: fuente.split()[-1] for clave, fuente in fichas_exogenas.loc[fichas_exogenas['tipo'] == 'actividad', 'Fuente'].items()}
sectores = {clave: descargar_bcrp(codigo) for clave, codigo in codigos_sectores.items()}
with open(ARCHIVO_DATOS, 'w', encoding='utf-8') as archivo:
    json.dump(datos_web, archivo, ensure_ascii=False)

instalar('holidays', 'holidays')
import holidays

feriados = np.array(list(holidays.country_holidays('PE', years=range(1994, 2028))), dtype='datetime64[D]')


def calendario(fechas):
    fechas = pd.DatetimeIndex(fechas)
    inicio_mes = fechas.values.astype('datetime64[D]')
    fin_mes = (fechas + pd.offsets.MonthBegin(1)).values.astype('datetime64[D]')
    previos = {anio: pd.date_range(end=pd.Timestamp(easter(anio)), periods=9)[:-1] for anio in set(fechas.year)}
    return pd.DataFrame({'dias_habiles': np.busday_count(inicio_mes, fin_mes, holidays=feriados),
                         'semana_santa': [np.mean((previos[f.year].year == f.year) & (previos[f.year].month == f.month))
                                          for f in fechas]}, index=fechas)


for nombre, s in [('Índice PBI Electricidad y Agua', indice), ('Variación % interanual', variacion),
                  ('Precio del cobre', cobre), ('ICEN', icen),
                  *[(fichas_exogenas.loc[clave, 'Variable'], s) for clave, s in sectores.items()]]:
    s = s.dropna()
    print(f'{nombre:<32} {s.index.min():%Y-%m} a {s.index.max():%Y-%m} ({len(s)} datos)')

control = (100 * (indice / indice.shift(12) - 1) - variacion).abs().max()
print(f'\nControl: la variación interanual calculada con el índice coincide con PN01723AM (diferencia máxima {control:.1e})')

## 3. Dataset y preparación

### 3.1. Tratamiento de la pandemia

In [ ]:
pandemia = pd.date_range('2020-03-01', '2021-06-01', freq='MS')

def imputar_pandemia(indice):
    z = np.log(indice)
    estacional = seasonal_decompose(z[:'2019-12-01'], model='additive', period=12).seasonal
    factor = estacional.groupby(estacional.index.month).mean()
    desestacionalizado = z - factor.reindex(z.index.month).values
    desestacionalizado[pandemia] = np.nan
    desestacionalizado = desestacionalizado.interpolate(method='linear')
    z_imputado = z.copy()
    z_imputado[pandemia] = desestacionalizado[pandemia] + factor.reindex(pandemia.month).values
    return np.exp(z_imputado)

indice_original = indice
indice = imputar_pandemia(indice_original)
display(pd.DataFrame({'Original': indice_original[pandemia].values, 'Imputado': indice[pandemia].values},
                     index=pandemia.strftime('%Y-%m')).round(1).T)
ventana = slice('2019-06-01', '2021-12-01')
print('Empalme: julio 2021 (real) vs julio 2019 (real):',
      f"{100 * (indice['2021-07-01'] / indice['2019-07-01'] - 1):.1f} % en dos años")

### 3.2. Transformación logarítmica

In [ ]:
inicio_muestra = pd.Timestamp('2002-01-01')
indice, indice_original = indice[inicio_muestra:], indice_original[inicio_muestra:]
serie = (100 * np.log(indice)).rename('100·ln(índice)')
cobre_log = (100 * np.log(cobre)).rename('100·ln(cobre)')
calendario_mensual = calendario(pd.date_range('1994-01-01', FIN, freq='MS'))
fuentes_exogenas = {**{clave: 100 * np.log(s) for clave, s in sectores.items()}, 'cobre': cobre_log, 'icen': icen,
                    **{clave: calendario_mensual[clave] for clave in calendario_mensual}}
candidatas = {clave: fuentes_exogenas[clave].rename(clave) for clave in fichas_exogenas.index}

datos = pd.concat([indice.rename('Índice PBI Electricidad y Agua'), serie, cobre.rename('Cobre (¢US$/lb)'),
                   cobre_log, icen], axis=1).loc[serie.index]
print('Periodo analizado:', serie.index.min().strftime('%Y-%m'), 'a', serie.index.max().strftime('%Y-%m'))
print('Observaciones:', len(serie), '| Meses faltantes:', int(datos.isna().sum().sum()))
datos.head()

In [ ]:
estadisticas = datos.describe().T
estadisticas['asimetría'] = datos.skew()
estadisticas['curtosis'] = datos.kurtosis()
estadisticas.round(4)

In [ ]:
por_anio = pd.DataFrame({
    'Promedio del índice': indice.groupby(indice.index.year).mean(),
    'Desv. estándar en niveles': indice.groupby(indice.index.year).std(),
    'Desv. estándar en logaritmos (%)': serie.groupby(serie.index.year).std(),
})
display(por_anio.iloc[::4].round(2))
print('Correlación entre el nivel del índice y su dispersión anual:')
print(f'  en niveles:     {por_anio.iloc[:, 0].corr(por_anio.iloc[:, 1]):.3f}')
print(f'  en logaritmos:  {por_anio.iloc[:, 0].corr(por_anio.iloc[:, 2]):.3f}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(indice_original, color='gray', linewidth=1, label='Original')
axes[0].plot(indice, color='blue', linewidth=1, label='Con pandemia imputada')
axes[0].axvspan(pandemia[0], pandemia[-1], color='gray', alpha=0.2)
axes[0].set_title('Índice PBI Electricidad y Agua (2007 = 100)', fontweight='bold'); axes[0].legend()
axes[1].plot(serie, color='purple', linewidth=1)
axes[1].set_title('Serie transformada: 100 · ln(índice)', fontweight='bold')
for ax in axes:
    ax.grid(alpha=0.3); ax.set_xlim(serie.index.min(), serie.index.max())
plt.tight_layout(); plt.show()

### 3.3. Ploteo de la serie y de las variables exógenas

In [ ]:
colores_tipo_exog = {'actividad': 'steelblue', 'precio': 'darkorange', 'clima': 'teal', 'calendario': 'dimgray'}
fig, axes = plt.subplot_mosaic([['serie', 'serie']] + [[f'p{2 * i}', f'p{2 * i + 1}'] for i in range(4)],
                               figsize=(15, 15), sharex=True)
axes['serie'].plot(serie, color='purple', linewidth=1)
axes['serie'].set_title('Serie: 100 · ln(índice PBI Electricidad y Agua)', fontweight='bold')
for i, (clave, x) in enumerate(candidatas.items()):
    ficha = fichas_exogenas.loc[clave]
    ax = axes[f'p{i}']
    ax.plot(x[serie.index], color=colores_tipo_exog[ficha['tipo']], linewidth=1)
    transformacion = '' if ficha['Transformación'] == 'Ninguna' else f" ({ficha['Transformación']})"
    ax.set_title(f"Candidata {i + 1}: {ficha['Variable']}{transformacion}", fontweight='bold')
    if clave == 'icen':
        ax.axhline(0.4, color='red', linestyle=':', linewidth=1, label='Umbral cálido (+0,4 °C)')
        ax.axhline(-1.0, color='blue', linestyle=':', linewidth=1, label='Umbral frío (−1,0 °C)')
        ax.axhline(0, linestyle='--', color='black', linewidth=0.6)
        ax.legend(loc='upper left', fontsize=7)
for ax in axes.values():
    ax.grid(alpha=0.3)
axes['serie'].set_xlim(serie.index.min(), serie.index.max())
plt.tight_layout(); plt.show()

In [ ]:
descomposicion = seasonal_decompose(serie, model='additive', period=12)

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
descomposicion.trend.plot(ax=axes[0], color='green', title='Tendencia')
descomposicion.seasonal.plot(ax=axes[1], color='orange', title='Estacionalidad')
descomposicion.resid.plot(ax=axes[2], color='gray', title='Residuo')
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 3.4. Cuartiles y deciles

In [ ]:
variacion_muestra = variacion[inicio_muestra:]
variables_desc = pd.DataFrame({'Índice PBI Electricidad y Agua': indice_original,
                               'Variación % interanual (BCRP)': variacion_muestra,
                               'Cobre (¢US$/lb)': cobre[inicio_muestra:FIN],
                               'ICEN (°C)': icen[inicio_muestra:FIN],
                               **{fichas_exogenas.loc[clave, 'Variable']: s[inicio_muestra:FIN] for clave, s in sectores.items()}})

cuartiles = variables_desc.quantile([0.25, 0.50, 0.75]).T
cuartiles.columns = ['Q1 (P25)', 'Q2 = mediana (P50)', 'Q3 (P75)']
cuartiles['IQR = Q3 − Q1'] = cuartiles['Q3 (P75)'] - cuartiles['Q1 (P25)']
cuartiles['Media'] = variables_desc.mean()
print('Cuartiles (2002-01 a 2026-07, datos reales sin imputar):')
display(cuartiles.round(2))

deciles = variables_desc.quantile(np.arange(0.1, 1.0, 0.1)).T
deciles.columns = [f'D{i}' for i in range(1, 10)]
print('Deciles:')
display(deciles.round(2))

ultimo = variables_desc.iloc[-1]
decil_ultimo = pd.DataFrame({'Último dato (2026-07)': ultimo,
                             'Decil': [int(np.searchsorted(deciles.loc[v].values, ultimo[v], side='right')) + 1
                                       for v in variables_desc.columns],
                             'Percentil empírico (%)': [100 * (variables_desc[v] <= ultimo[v]).mean()
                                                        for v in variables_desc.columns]})
print('Posición del último dato (decil 1 = 10 % más bajo; decil 10 = 10 % más alto):')
display(decil_ultimo.round(2))

In [ ]:
def atipicos_iqr(x):
    q1, q3 = x.quantile([0.25, 0.75])
    iqr = q3 - q1
    inferior, superior = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return inferior, superior, x[(x < inferior) | (x > superior)]

filas = []
for nombre, x in [('Índice (nivel, con tendencia)', indice_original), ('Variación % interanual (estacionaria)', variacion_muestra)]:
    inferior, superior, fuera = atipicos_iqr(x)
    filas.append({'Serie': nombre, 'Límite inferior': inferior, 'Límite superior': superior,
                  'N.º de atípicos': len(fuera), '% de atípicos': 100 * len(fuera) / len(x)})
display(pd.DataFrame(filas).set_index('Serie').round(2))

inferior, superior, atipicos_var = atipicos_iqr(variacion_muestra)
print('Meses atípicos de la variación % interanual (regla 1,5·IQR):')
display(atipicos_var.round(1).to_frame('Var. %').T.rename(columns=lambda f: f.strftime('%Y-%m')))

q1, q2, q3 = variacion_muestra.quantile([0.25, 0.5, 0.75])
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={'width_ratios': [1, 4]})
axes[0].boxplot(variacion_muestra, widths=0.5)
axes[0].set_xticks([1], ['Var. % interanual'])
axes[0].set_title('Diagrama de caja', fontweight='bold')
axes[1].plot(variacion_muestra, color='gray', linewidth=0.9)
axes[1].axhspan(q1, q3, color='steelblue', alpha=0.2, label=f'Entre Q1 y Q3 ({q1:.1f} % a {q3:.1f} %)')
axes[1].axhline(q2, color='steelblue', linewidth=1, label=f'Mediana ({q2:.1f} %)')
axes[1].axhline(inferior, color='red', linestyle=':', linewidth=1, label='Límites 1,5·IQR')
axes[1].axhline(superior, color='red', linestyle=':', linewidth=1)
axes[1].scatter(atipicos_var.index, atipicos_var, color='red', zorder=3, label='Atípicos')
axes[1].set_title('Variación % interanual del PBI Electricidad y Agua: cuartiles y atípicos', fontweight='bold')
axes[1].legend(loc='lower left', fontsize=8)
axes[1].set_xlim(variacion_muestra.index.min(), variacion_muestra.index.max())
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
crecimiento_mensual = (100 * np.log(indice).diff()).dropna()
nombres_mes = ['Ene', 'Feb', 'Mar', 'Abr', 'May', 'Jun', 'Jul', 'Ago', 'Set', 'Oct', 'Nov', 'Dic']
por_mes = crecimiento_mensual.groupby(crecimiento_mensual.index.month)
cuartiles_mes = por_mes.quantile([0.25, 0.5, 0.75]).unstack()
cuartiles_mes.columns = ['Q1', 'Mediana', 'Q3']
cuartiles_mes['IQR'] = cuartiles_mes['Q3'] - cuartiles_mes['Q1']
cuartiles_mes.index = nombres_mes
print('Crecimiento mensual (%, 100·Δln índice) por mes del año:')
display(cuartiles_mes.T.round(2))

plt.figure(figsize=(14, 4.5))
plt.boxplot([grupo.values for _, grupo in por_mes], widths=0.6)
plt.xticks(range(1, 13), nombres_mes)
plt.axhline(0, color='black', linestyle='--', linewidth=0.7)
plt.title('Crecimiento mensual del índice por mes del año (cuartiles)', fontweight='bold')
plt.ylabel('% respecto del mes anterior'); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Identificación y estacionariedad

### 4.1. FAC y FACP de la serie en logaritmos

In [ ]:
inicio_test = pd.Timestamp('2022-01-01')

def graficar_fac(x, titulo):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    plot_acf(x, lags=48, zero=False, ax=axes[0], title=f'FAC (q) — {titulo}')
    plot_pacf(x, lags=48, zero=False, method='ywm', ax=axes[1], title=f'FACP (p) — {titulo}')
    plt.tight_layout(); plt.show()

graficar_fac(serie[serie.index < inicio_test], '100·ln(índice) (train)')

### 4.2. ¿Es la serie estacionaria? Diferenciación regular y estacional

In [ ]:
def evaluar(x):
    x = x.dropna()
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        p_kpss = kpss(x, regression='c', nlags='auto')[1]
        p_adf = adfuller(x, autolag='AIC')[1]
    fac12 = acf(x, nlags=12)[12]
    cumple = p_adf < alpha and p_kpss >= alpha and fac12 < 0.5
    return {'p-valor ADF': p_adf, 'p-valor KPSS': p_kpss, 'FAC rezago 12': fac12,
            'Conclusión': 'Estacionaria' if cumple else 'No estacionaria'}

train_log = serie[serie.index < inicio_test]
transformaciones = {
    (0, 0): ('Logaritmo (d=0, D=0)', train_log),
    (1, 0): ('Logaritmo + 1.ª diferencia (d=1, D=0)', train_log.diff()),
    (0, 1): ('Logaritmo + diferencia estacional (d=0, D=1)', train_log.diff(12)),
    (1, 1): ('Logaritmo + ambas diferencias (d=1, D=1)', train_log.diff().diff(12)),
}
pruebas_est = pd.DataFrame({nombre: evaluar(x) for nombre, x in transformaciones.values()}).T
display(pruebas_est.style.format({'p-valor ADF': '{:.4f}', 'p-valor KPSS': '{:.4f}', 'FAC rezago 12': '{:.4f}'}))

d, D = next(clave for clave, (nombre, x) in transformaciones.items() if evaluar(x)['Conclusión'] == 'Estacionaria')
print(f'Orden de diferenciación elegido: d = {d}, D = {D} (s = 12)')

### 4.3. Serie transformada y su FAC y FACP

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
for (nombre, x), ax in zip(transformaciones.values(), axes):
    x.dropna().plot(ax=ax, color='blue', linewidth=0.8)
    ax.set_title(nombre, fontweight='bold'); ax.grid(alpha=0.3)
axes[0].set_xlim(train_log.index.min(), train_log.index.max())
plt.tight_layout(); plt.show()

estacionaria_train = transformaciones[(d, D)][1].dropna()
graficar_fac(estacionaria_train, transformaciones[(d, D)][0])

## 5. División de la muestra en train y test

In [ ]:
train, test = serie[serie.index < inicio_test], serie[serie.index >= inicio_test]
indice_test = indice[test.index]

print('Train:', train.index.min().strftime('%Y-%m'), 'a', train.index.max().strftime('%Y-%m'), f'({len(train)} datos, {len(train) / len(serie):.0%})')
print('Test: ', test.index.min().strftime('%Y-%m'), 'a', test.index.max().strftime('%Y-%m'), f'({len(test)} datos, {len(test) / len(serie):.0%})')

## 6. Identificación del modelo y estimación de parámetros

### 6.1. Modelos univariados: AR, MA, ARMA, ARIMA y SARIMA

In [ ]:
def tendencia_de(orden, estacional):
    n_dif = orden[1] + estacional[1]
    return 'n' if n_dif >= 2 else 't' if n_dif == 1 else 'c'

tendencia = tendencia_de((0, d, 0), (0, D, 0, 12))

def ajustar(datos_y, orden, estacional, exog=None):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return ARIMA(datos_y, exog=exog, order=orden, seasonal_order=estacional,
                     trend=tendencia_de(orden, estacional)).fit(method_kwargs={'maxiter': 500})

def n_param(orden, estacional):
    return orden[0] + orden[2] + estacional[0] + estacional[2]

def familia(orden, estacional):
    if estacional[0] + estacional[2] > 0:
        return 'SARIMA'
    if estacional[1] == 0:
        return 'ARIMA'
    return 'AR' if orden[2] == 0 else 'MA' if orden[0] == 0 else 'ARMA'

def nombre_modelo(orden, estacional):
    if familia(orden, estacional) == 'ARIMA':
        return f'ARIMA{orden}'
    return f'{familia(orden, estacional)}{orden}{estacional[:3]}12'

def residuos_validos(m):
    return m.resid.iloc[m.model.k_diff + m.model.seasonal_periods * m.model.k_seasonal_diff:]

pq, PQ = range(CONFIG['sarima_pq_max'] + 1), range(CONFIG['sarima_PQ_max'] + 1)
ordenes = [((p, d, q), (P, D, Q, 12)) for p in pq for q in pq for P in PQ for Q in PQ
           if p + q + P + Q > 0 and (P + Q == 0 or p + q <= CONFIG['sarima_pq_suma_max'])]

modelos, resultados = {}, []
for orden, estacional in ordenes:
    clave = nombre_modelo(orden, estacional)
    try:
        ajuste = ajustar(train, orden, estacional)
    except (ValueError, np.linalg.LinAlgError):
        continue
    if not ajuste.mle_retvals.get('converged', True) or not np.isfinite(ajuste.bic):
        continue
    lb = acorr_ljungbox(residuos_validos(ajuste), lags=[12, 24], model_df=n_param(orden, estacional))['lb_pvalue']
    modelos[clave] = (orden, estacional)
    resultados.append({'Modelo': clave, 'Familia': familia(orden, estacional), 'AIC': ajuste.aic, 'BIC': ajuste.bic,
                       'p-Ljung-Box(12)': lb.iloc[0], 'p-Ljung-Box(24)': lb.iloc[1]})

comparacion = pd.DataFrame(resultados).sort_values('BIC').reset_index(drop=True)
comparacion['Pasa Ljung-Box'] = np.where(comparacion[['p-Ljung-Box(12)', 'p-Ljung-Box(24)']].min(axis=1) >= alpha, 'Sí', 'No')
print(f'Modelos estimados: {len(comparacion)}. Se muestran los 15 de menor BIC:')
display(comparacion.head(15).round(4))

In [ ]:
mejores = (comparacion.sort_values(['Pasa Ljung-Box', 'BIC'], ascending=[False, True])
           .groupby('Familia', sort=False).head(1).set_index('Familia').loc[['AR', 'MA', 'ARMA', 'SARIMA']])
display(mejores.round(4))

nombre_sarima = mejores.loc['SARIMA', 'Modelo']
orden, estacional = modelos[nombre_sarima]
k = n_param(orden, estacional)
modelo_sarima = ajustar(train, orden, estacional)
print(f'Orden base para el SARIMAX: {nombre_sarima}')
display(pd.DataFrame({'Coeficiente': modelo_sarima.params, 'p-valor': modelo_sarima.pvalues}).round(4))

#### b) ARIMA(p, d, q) sin parte estacional

In [ ]:
d_arima = max(d, 1)
ordenes_arima = [((p, d_arima, q), (0, 0, 0, 0)) for p in range(CONFIG['arima_pq_max'] + 1)
                 for q in range(CONFIG['arima_pq_max'] + 1) if p + q > 0]

resultados_arima = []
for orden_a, estacional_a in ordenes_arima:
    clave = nombre_modelo(orden_a, estacional_a)
    try:
        ajuste = ajustar(train, orden_a, estacional_a)
    except (ValueError, np.linalg.LinAlgError):
        continue
    if not ajuste.mle_retvals.get('converged', True) or not np.isfinite(ajuste.bic):
        continue
    lb_a = acorr_ljungbox(residuos_validos(ajuste), lags=[12, 24], model_df=n_param(orden_a, estacional_a))['lb_pvalue']
    modelos[clave] = (orden_a, estacional_a)
    resultados_arima.append({'Modelo': clave, 'Familia': 'ARIMA', 'AIC': ajuste.aic, 'BIC': ajuste.bic,
                             'p-Ljung-Box(12)': lb_a.iloc[0], 'p-Ljung-Box(24)': lb_a.iloc[1]})

comparacion_arima = pd.DataFrame(resultados_arima).sort_values('BIC').reset_index(drop=True)
comparacion_arima['Pasa Ljung-Box'] = np.where(
    comparacion_arima[['p-Ljung-Box(12)', 'p-Ljung-Box(24)']].min(axis=1) >= alpha, 'Sí', 'No')
print(f'Modelos ARIMA estimados: {len(comparacion_arima)} (AIC y BIC comparables solo entre ellos)')
display(comparacion_arima.round(4))

mejor_arima = (comparacion_arima.sort_values(['Pasa Ljung-Box', 'BIC'], ascending=[False, True])
               .head(1).set_index('Familia'))
mejores = pd.concat([mejores.drop(index='ARIMA', errors='ignore'), mejor_arima]).loc[['AR', 'MA', 'ARMA', 'ARIMA', 'SARIMA']]
nombre_arima = mejores.loc['ARIMA', 'Modelo']
modelo_arima = ajustar(train, *modelos[nombre_arima])
print(f'Mejor ARIMA: {nombre_arima}')
display(mejores.round(4))
display(pd.DataFrame({'Coeficiente': modelo_arima.params, 'p-valor': modelo_arima.pvalues}).round(4))

In [ ]:
def guardar_figura(nombre):
    os.makedirs('Image', exist_ok=True)
    plt.savefig(os.path.join('Image', nombre), dpi=200, bbox_inches='tight')

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
for fila, (nombre_m, m) in enumerate([(nombre_arima, modelo_arima), (nombre_sarima, modelo_sarima)]):
    r = residuos_validos(m)
    plot_acf(r, lags=36, zero=False, ax=axes[fila, 0], title=f'FAC de residuos - {nombre_m}')
    plot_pacf(r, lags=36, zero=False, method='ywm', ax=axes[fila, 1], title=f'FACP de residuos - {nombre_m}')
    for ax in axes[fila]:
        for rezago in (12, 24, 36):
            ax.axvline(rezago, color='red', linestyle=':', linewidth=0.8)
plt.tight_layout(); guardar_figura('ts_06b_arima_fac_residuos.png'); plt.show()

fac_arima, fac_sarima = acf(residuos_validos(modelo_arima), nlags=24), acf(residuos_validos(modelo_sarima), nlags=24)
display(pd.DataFrame({f'FAC residuos {nombre_arima}': fac_arima[[1, 12, 24]],
                      f'FAC residuos {nombre_sarima}': fac_sarima[[1, 12, 24]]},
                     index=['Rezago 1', 'Rezago 12', 'Rezago 24']).round(3))

### 6.2. Variables exógenas: candidatas y selección automática
#### a) Variables candidatas

In [ ]:
display(fichas_exogenas.drop(columns=['Corto', 'tipo']))

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for ax, (inicio, fin) in zip(axes, [('2016-07-01', '2018-06-01'), ('2022-10-01', '2024-12-01')]):
    tramo = slice(inicio, fin)
    pico = icen[tramo].idxmax()
    ax.bar(icen[tramo].index, icen[tramo], width=25, color=np.where(icen[tramo] > 0.4, 'tomato', 'lightgray'), label='ICEN (°C)')
    ax2 = ax.twinx()
    ax2.plot(variacion[tramo], color='black', marker='o', markersize=3, label='Var. % interanual del sector')
    ax2.axhline(0, color='gray', linewidth=0.6)
    ax.axvline(pico, color='red', linestyle=':', linewidth=1)
    ax.axvspan(pico + pd.DateOffset(months=8), pico + pd.DateOffset(months=11), color='orange', alpha=0.2,
               label='8 a 11 meses después del pico')
    ax.set_title(f'Pico del ICEN: {pico:%Y-%m} ({icen[pico]:.2f} °C)', fontweight='bold')
    ax.set_ylabel('ICEN (°C)'); ax2.set_ylabel('Var. % interanual')
    lineas = ax.get_legend_handles_labels(); lineas2 = ax2.get_legend_handles_labels()
    ax.legend(lineas[0] + lineas2[0], lineas[1] + lineas2[1], loc='upper right', fontsize=8)
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

resumen_eventos = pd.DataFrame({
    'Promedio 2002–2019': variacion['2002':'2019'].mean(),
    '2017': variacion['2017'].mean(), '2018': variacion['2018'].mean(),
    '2023': variacion['2023'].mean(), '2024': variacion['2024'].mean()}, index=['Var. % interanual promedio'])
display(resumen_eventos.round(2))

#### b) Selección del rezago

In [ ]:
def diferenciar(x):
    x = x.diff(1) if d else x
    return x.diff(12) if D else x

rezagos_candidata = {clave: range(int(f['Rezago mínimo']), 1 if f['tipo'] == 'calendario' else CONFIG['rezago_max_exog'] + 1)
                     for clave, f in fichas_exogenas.iterrows()}

y_dif = diferenciar(serie)[train.index].dropna()
banda = 1.96 / np.sqrt(len(y_dif))
fig, axes = plt.subplots(2, 4, figsize=(18, 7), sharey=True)
for ax, (clave, x) in zip(axes.flat, candidatas.items()):
    x_dif = diferenciar(x)
    corr = [y_dif.corr(x_dif.shift(L).reindex(y_dif.index)) for L in range(CONFIG['rezago_max_exog'] + 1)]
    ax.bar(range(CONFIG['rezago_max_exog'] + 1), corr, color=['steelblue' if L in rezagos_candidata[clave] else 'lightgray' for L in range(CONFIG['rezago_max_exog'] + 1)])
    ax.axhline(banda, color='red', linestyle='--', linewidth=0.8); ax.axhline(-banda, color='red', linestyle='--', linewidth=0.8)
    ax.set_title(fichas_exogenas.loc[clave, 'Variable'], fontweight='bold')
    ax.set_xlabel('Rezago (meses)'); ax.grid(alpha=0.3)
plt.suptitle('Correlación cruzada de la serie con cada candidata, ambas diferenciadas (gris = rezago no probado)', fontweight='bold')
plt.tight_layout(); plt.show()

filas = []
for clave, x in candidatas.items():
    for L in rezagos_candidata[clave]:
        m = ajustar(train, orden, estacional, exog=x.shift(L).reindex(train.index))
        filas.append({'Variable': clave, 'Rezago': L, 'BIC': m.bic, 'Δ BIC vs sin exógena': m.bic - modelo_sarima.bic,
                      'Coeficiente': m.params[clave], 'p-valor': m.pvalues[clave]})
busqueda = pd.DataFrame(filas)
rezago_elegido = busqueda.loc[busqueda.groupby('Variable', sort=False)['BIC'].idxmin()].set_index('Variable')
display(busqueda.pivot(index='Rezago', columns='Variable', values='BIC')[list(candidatas)]
        .rename(columns=fichas_exogenas['Variable']).style.format('{:.2f}', na_rep='—')
        .highlight_min(color='#c6efce', axis=0).set_caption('BIC del SARIMAX con una sola candidata (verde = rezago elegido)'))
print('Rezago elegido (menor BIC):')
display(rezago_elegido.rename(index=fichas_exogenas['Variable']).round(4))

In [ ]:
X = pd.DataFrame(index=serie.index)
rezago_exog, etiqueta_col = {}, {}
for clave, fila in rezago_elegido.iterrows():
    L = int(fila['Rezago'])
    col = f'{clave}_t-{L}' if L else clave
    X[col] = candidatas[clave].shift(L).reindex(serie.index)
    rezago_exog[col] = (clave, L)
    etiqueta_col[col] = f"{fichas_exogenas.loc[clave, 'Variable']} (t−{L})" if L else fichas_exogenas.loc[clave, 'Variable']
columna_de = {clave: col for col, (clave, _) in rezago_exog.items()}
X.head()

#### c) Filtro de colinealidad

In [ ]:
UMBRAL_CORR, UMBRAL_VIF = 0.8, 5
X_dif = diferenciar(X).loc[y_dif.index]
corr_exog = X_dif.corr()
orden_bic = rezago_elegido['BIC'].sort_values().index


def vif(columnas):
    matriz = np.column_stack([np.ones(len(X_dif)), X_dif[columnas].to_numpy()])
    return pd.Series([variance_inflation_factor(matriz, i + 1) for i in range(len(columnas))], index=columnas)


conservadas, decision = [], {}
for clave in orden_bic:
    col = columna_de[clave]
    choque = corr_exog.loc[col, conservadas].abs()
    if len(choque) and choque.max() > UMBRAL_CORR:
        decision[col] = f'Se descarta: |r| = {choque.max():.2f} con {etiqueta_col[choque.idxmax()]}'
    else:
        conservadas.append(col)
        decision[col] = 'Pasa al paso d)'

vif_final = vif(conservadas)
while len(conservadas) > 1 and vif_final.max() > UMBRAL_VIF:
    peor = max(vif_final[vif_final > UMBRAL_VIF].index, key=lambda col: rezago_elegido.loc[rezago_exog[col][0], 'BIC'])
    decision[peor] = f'Se descarta: VIF = {vif_final[peor]:.1f}'
    conservadas.remove(peor)
    vif_final = vif(conservadas)

etiquetas = [etiqueta_col[col] for col in corr_exog.columns]
fig, ax = plt.subplots(figsize=(10, 8))
imagen = ax.imshow(corr_exog, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(etiquetas)), etiquetas, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(etiquetas)), etiquetas, fontsize=8)
for i in range(len(etiquetas)):
    for j in range(len(etiquetas)):
        valor = corr_exog.iloc[i, j]
        ax.text(j, i, f'{valor:.2f}', ha='center', va='center', fontsize=8, color='white' if abs(valor) > 0.6 else 'black')
fig.colorbar(imagen, ax=ax, fraction=0.046)
ax.set_title(f'Correlación entre candidatas con su rezago, diferenciadas (d = {d}, D = {D})\nUmbral de descarte: |r| > {UMBRAL_CORR}',
             fontweight='bold')
plt.tight_layout(); guardar_figura('ts_06c_correlacion_exogenas.png'); plt.show()

filtro_colinealidad = pd.DataFrame({
    'Variable': [etiqueta_col[columna_de[clave]] for clave in orden_bic],
    'BIC individual': rezago_elegido.loc[orden_bic, 'BIC'].values,
    'p-valor individual': rezago_elegido.loc[orden_bic, 'p-valor'].values,
    'Decisión': [decision[columna_de[clave]] for clave in orden_bic],
    'VIF final': [vif_final.get(columna_de[clave], np.nan) for clave in orden_bic],
}).set_index('Variable')
display(filtro_colinealidad.style.format({'BIC individual': '{:.2f}', 'p-valor individual': '{:.4f}', 'VIF final': '{:.2f}'},
                                         na_rep='—')
        .set_caption(f'Filtro de colinealidad: orden por BIC individual, |r| ≤ {UMBRAL_CORR} y VIF ≤ {UMBRAL_VIF}'))

#### d) Eliminación hacia atrás en el SARIMAX conjunto

In [ ]:
seleccion, pasos_eliminacion = list(conservadas), []
while seleccion:
    m = ajustar(train, orden, estacional, exog=X.loc[train.index, seleccion])
    p_valores = m.pvalues[seleccion]
    peor = p_valores.idxmax()
    pasos_eliminacion.append({'Paso': len(pasos_eliminacion) + 1, 'Variables': ', '.join(etiqueta_col[col] for col in seleccion),
                              'BIC': m.bic, 'Δ BIC vs SARIMA': m.bic - modelo_sarima.bic, 'Mayor p-valor': p_valores[peor],
                              'Acción': 'Fin: todas significativas' if p_valores[peor] < alpha else f'Se elimina {etiqueta_col[peor]}'})
    if p_valores[peor] < alpha:
        break
    seleccion.remove(peor)

respaldo = not seleccion
if respaldo:
    seleccion = [min(conservadas, key=lambda col: rezago_elegido.loc[rezago_exog[col][0], 'p-valor'])]
display(pd.DataFrame(pasos_eliminacion).set_index('Paso').style
        .format({'BIC': '{:.2f}', 'Δ BIC vs SARIMA': '{:+.2f}', 'Mayor p-valor': '{:.4f}'})
        .set_caption(f'Eliminación hacia atrás en el SARIMAX conjunto (α = {alpha})'))
print('Variables seleccionadas:', ', '.join(etiqueta_col[col] for col in seleccion))
if respaldo:
    print(f'Ninguna candidata es significativa al {alpha:.0%} en el modelo conjunto: se conserva la de menor p-valor individual.')

### 6.3. Estimación de los modelos SARIMAX

In [ ]:
cols_exog = seleccion
etiqueta_exog = ' + '.join(fichas_exogenas.loc[rezago_exog[col][0], 'Corto'] for col in cols_exog)
nombre_sarimax = f'SARIMAX + {etiqueta_exog}'
conjuntos = {nombre_sarimax: cols_exog}
if len(cols_exog) > 1:
    conjuntos.update({f"SARIMAX + {fichas_exogenas.loc[rezago_exog[col][0], 'Corto']}": [col] for col in cols_exog})
sarimax = {etiqueta: ajustar(train, orden, estacional, exog=X.loc[train.index, cols]) for etiqueta, cols in conjuntos.items()}

for etiqueta, m in sarimax.items():
    lb = acorr_ljungbox(residuos_validos(m), lags=[12, 24], model_df=k)['lb_pvalue']
    print(f'\n{etiqueta} {orden}{estacional[:3]}12 | AIC {m.aic:.2f} | BIC {m.bic:.2f} | Δ BIC vs SARIMA {m.bic - modelo_sarima.bic:+.2f} '
          f'| p-Ljung-Box(12, 24) = {lb.iloc[0]:.4f}, {lb.iloc[1]:.4f}')
    display(pd.DataFrame({'Coeficiente': m.params, 'Error estándar': m.bse, 'p-valor': m.pvalues}).round(4))

modelo = sarimax[nombre_sarimax]
print(f'Modelo propuesto: {nombre_sarimax} | SARIMA sin exógenas: AIC {modelo_sarima.aic:.2f}, BIC {modelo_sarima.bic:.2f}')

## 7. Pronóstico y contraste con los valores reales

In [ ]:
candidatos = {f'{familia} - {mejores.loc[familia, "Modelo"]}': (modelos[mejores.loc[familia, 'Modelo']], None)
              for familia in ['AR', 'MA', 'ARMA', 'ARIMA', 'SARIMA']}
for etiqueta, cols in conjuntos.items():
    candidatos[f'{etiqueta} - {orden}{estacional[:3]}12'] = ((orden, estacional), cols)

def a_indice(z):
    return np.exp(z / 100)

def a_variacion(indice_pred, indice_base):
    base = indice_base.reindex(indice_pred.index - pd.DateOffset(years=1)).values
    base = np.where(np.isnan(base), indice_pred.shift(12).values, base)
    return pd.Series(100 * (indice_pred.values / base - 1), index=indice_pred.index)

def metricas(real, pred):
    return {'MSE': mean_squared_error(real, pred), 'RMSE': np.sqrt(mean_squared_error(real, pred)),
            'MAE': mean_absolute_error(real, pred), 'MAPE (%)': np.mean(np.abs((real - pred) / real)) * 100}

indice_train = indice[train.index]
variacion_test = variacion[test.index]

pronosticos, pronosticos_1m, bandas, ajustes = {}, {}, {}, {}
for etiqueta, ((o, e), cols) in candidatos.items():
    m = ajustar(train, o, e, exog=X.loc[train.index, cols] if cols else None)
    f = m.get_forecast(steps=len(test), exog=X.loc[test.index, cols] if cols else None)
    pronosticos[etiqueta] = a_indice(f.predicted_mean)
    bandas[etiqueta] = a_indice(f.conf_int(alpha=0.05))
    actualizado = m.apply(serie, exog=X.loc[serie.index, cols] if cols else None)
    pronosticos_1m[etiqueta] = a_indice(actualizado.get_prediction(start=test.index[0], end=test.index[-1]).predicted_mean)
    ajustes[etiqueta] = (m, n_param(o, e))

ultimo_crecimiento = indice_original[train.index].iloc[-1] / indice_original[train.index].iloc[-13]
anios = np.arange(len(test)) // 12 + 1
mismo_mes_2021 = indice_original[train.index].iloc[-12:].values[np.arange(len(test)) % 12]
pronosticos['Ingenuo estacional'] = pd.Series(mismo_mes_2021, index=test.index)
pronosticos['Ingenuo con último crecimiento'] = pd.Series(mismo_mes_2021 * ultimo_crecimiento ** anios, index=test.index)
pronosticos_1m['Ingenuo estacional'] = indice_original.shift(12)[test.index]
pronosticos_1m['Ingenuo con último crecimiento'] = (indice_original.shift(12) * indice_original.shift(1) / indice_original.shift(13))[test.index]

test_consolidado = pd.concat({'Real': indice_test, **pronosticos}, axis=1)
test_consolidado.head().round(2)

### 7.1. Ploteo de los pronósticos

In [ ]:
etiqueta_final = f'{nombre_sarimax} - {orden}{estacional[:3]}12'
etiqueta_sarima = f'SARIMA - {nombre_sarima}'
mostrar = [etiqueta_sarima, etiqueta_final, 'Ingenuo con último crecimiento']

fig, axes = plt.subplots(3, 1, figsize=(14, 14))
axes[0].plot(indice['2018-01-01':], color='black', linewidth=1.4, label='Real')
for etiqueta in mostrar:
    axes[0].plot(pronosticos[etiqueta], linestyle='-' if etiqueta == etiqueta_final else '--', label=etiqueta)
axes[0].fill_between(test.index, bandas[etiqueta_final].iloc[:, 0], bandas[etiqueta_final].iloc[:, 1],
                     color='red', alpha=0.1, label=f'IC 95 % ({etiqueta_exog})')
axes[0].set_title('Índice: pronóstico de todo el test desde dic-2021 (horizonte 1 a 55 meses)', fontweight='bold')

axes[1].plot(indice['2018-01-01':], color='black', linewidth=1.4, label='Real')
for etiqueta in mostrar:
    axes[1].plot(pronosticos_1m[etiqueta], linestyle='-' if etiqueta == etiqueta_final else '--', label=etiqueta)
axes[1].set_title('Índice: pronóstico a 1 mes, actualizando la serie cada mes', fontweight='bold')

variacion_real = variacion
axes[2].plot(variacion_real['2022-01-01':], color='black', linewidth=1.4, label='Real (BCRP)')
for etiqueta in mostrar:
    axes[2].plot(a_variacion(pronosticos_1m[etiqueta], indice_original), linestyle='-' if etiqueta == etiqueta_final else '--', label=etiqueta)
axes[2].axhline(0, color='gray', linewidth=0.6)
axes[2].set_title('Variación % interanual implícita en el pronóstico a 1 mes', fontweight='bold')
for ax in axes:
    ax.axvspan(test.index[0], test.index[-1], color='lightgray', alpha=0.4)
    ax.grid(alpha=0.3); ax.legend(loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

### 7.2. Tabla de métricas de los mejores modelos

In [ ]:
filas = []
for etiqueta in pronosticos:
    fila = {'Modelo': etiqueta}
    if etiqueta in ajustes:
        m, kk = ajustes[etiqueta]
        fila.update({'AIC': m.aic, 'BIC': m.bic,
                     'p-Ljung-Box(24)': acorr_ljungbox(residuos_validos(m), lags=[24], model_df=kk)['lb_pvalue'].iloc[0]})
    fila.update(metricas(indice_test, pronosticos[etiqueta]))
    fila['RMSE var. % (pp)'] = np.sqrt(mean_squared_error(variacion_test, a_variacion(pronosticos[etiqueta], indice_original[train.index])))
    m1 = metricas(indice_test, pronosticos_1m[etiqueta])
    fila.update({f'{c} 1 mes': m1[c] for c in ['RMSE', 'MAE', 'MAPE (%)']})
    fila['RMSE var. % (pp) 1 mes'] = np.sqrt(mean_squared_error(variacion_test, a_variacion(pronosticos_1m[etiqueta], indice_original)))
    filas.append(fila)

resumen_metricas = pd.DataFrame(filas).set_index('Modelo')
resumen_metricas.columns = pd.MultiIndex.from_tuples(
    [('Ajuste (train)', c) for c in ['AIC', 'BIC', 'p-Ljung-Box(24)']] +
    [('Test - horizonte completo', c) for c in ['MSE', 'RMSE', 'MAE', 'MAPE (%)', 'RMSE var. % (pp)']] +
    [('Test - 1 mes', c) for c in ['RMSE 1 mes', 'MAE 1 mes', 'MAPE (%) 1 mes', 'RMSE var. % (pp) 1 mes']])
(resumen_metricas.style.format('{:.4f}', na_rep='—')
 .highlight_min(subset=[c for c in resumen_metricas.columns if c[1] != 'p-Ljung-Box(24)'], color='#c6efce')
 .set_caption('Métricas de los mejores modelos (verde = mejor valor de cada columna)'))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
tabla_plana = resumen_metricas.droplevel(0, axis=1)
for ax, columna in zip(axes.flat, ['RMSE', 'MAPE (%)', 'RMSE 1 mes', 'MAPE (%) 1 mes']):
    valores = tabla_plana[columna].sort_values()
    colores = ['crimson' if e == etiqueta_final else 'gray' if 'Ingenuo' in e else 'steelblue' for e in valores.index]
    ax.barh([e.split(' - ')[0] for e in valores.index], valores.values, color=colores)
    ax.set_title(columna, fontweight='bold'); ax.invert_yaxis(); ax.grid(alpha=0.3, axis='x')
    for i, v in enumerate(valores.values):
        ax.text(v, i, f' {v:.2f}', va='center', fontsize=8)
plt.suptitle('Comparación de métricas en test sobre el índice (rojo = modelo propuesto, gris = ingenuos)', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
def diebold_mariano(real, pred_a, pred_b):
    dif = (real - pred_a)**2 - (real - pred_b)**2
    dm = dif.mean() / (dif.std() / np.sqrt(len(dif)))
    return dm, 2 * stats.norm.sf(abs(dm))

dm_tabla = []
for rival in [etiqueta_sarima, 'Ingenuo estacional', 'Ingenuo con último crecimiento']:
    dm, p = diebold_mariano(indice_test, pronosticos_1m[etiqueta_final], pronosticos_1m[rival])
    dm_tabla.append({'Comparación (1 mes)': f'Propuesto vs {rival.split(" - ")[0]}', 'Estadístico DM': dm, 'p-valor': p,
                     'Conclusión': 'Sin diferencia significativa' if p >= alpha else
                                   'El propuesto es significativamente mejor' if dm < 0 else 'El propuesto es significativamente peor'})
display(pd.DataFrame(dm_tabla).round(4))

### 7.3. ARIMA frente a los modelos estacionales

In [ ]:
etiqueta_arima = f'ARIMA - {nombre_arima}'
filas_73 = [f'{f} - {mejores.loc[f, "Modelo"]}' for f in ['AR', 'MA', 'ARMA', 'ARIMA', 'SARIMA']]
filas_73 += [etiqueta_final, 'Ingenuo estacional', 'Ingenuo con último crecimiento']
tabla_73 = tabla_plana.loc[filas_73, ['p-Ljung-Box(24)', 'RMSE', 'MAPE (%)', 'RMSE 1 mes', 'MAE 1 mes',
                                      'MAPE (%) 1 mes', 'RMSE var. % (pp) 1 mes']].copy()
tabla_73.insert(0, 'Puesto por RMSE a 1 mes', tabla_plana['RMSE 1 mes'].rank().loc[filas_73].astype(int))
display(tabla_73.style.format('{:.4f}', na_rep='—', subset=tabla_73.columns.drop('Puesto por RMSE a 1 mes')))

dm_a, p_a = diebold_mariano(indice_test, pronosticos_1m[etiqueta_sarima], pronosticos_1m[etiqueta_arima])
rmse_a, rmse_s = tabla_plana.loc[etiqueta_arima, 'RMSE 1 mes'], tabla_plana.loc[etiqueta_sarima, 'RMSE 1 mes']
print(f'RMSE a 1 mes: ARIMA {rmse_a:.2f} vs SARIMA {rmse_s:.2f} '
      f'(la parte estacional cambia el error en {100 * (rmse_s / rmse_a - 1):+.1f} %)')
print(f'Diebold–Mariano (SARIMA vs ARIMA): estadístico {dm_a:.3f}, p-valor {p_a:.4f} → '
      + ('sin diferencia significativa' if p_a >= alpha else
         'el SARIMA es significativamente mejor' if dm_a < 0 else 'el ARIMA es significativamente mejor'))

### 7.4. Prophet

In [ ]:
instalar('prophet', 'prophet')
logging.getLogger('prophet.plot').disabled = True
from prophet import Prophet
from prophet.utilities import regressor_coefficients
from cmdstanpy.utils import get_logger
get_logger().setLevel(logging.WARNING)
logging.getLogger('prophet').setLevel(logging.WARNING)

datos_prophet = pd.DataFrame({'ds': serie.index, 'y': serie.values, **{col: X.loc[serie.index, col].values for col in cols_exog}})
nombre_prophet_exog = f'Prophet + {etiqueta_exog}'
regresores_prophet = {'Prophet': [], nombre_prophet_exog: cols_exog}


def crear_prophet(regresores=(), changepoint_prior_scale=0.05, seasonality_prior_scale=10.0):
    m = Prophet(growth='linear', yearly_seasonality=False, weekly_seasonality=False, daily_seasonality=False,
                changepoint_prior_scale=changepoint_prior_scale, seasonality_prior_scale=seasonality_prior_scale,
                interval_width=0.95)
    m.add_seasonality(name='anual', period=365.25, fourier_order=6)
    for r in regresores:
        m.add_regressor(r)
    return m


def prophet_a_1_mes(fechas, regresores=(), **params):
    pred = {}
    for fecha in fechas:
        m = crear_prophet(regresores, **params).fit(datos_prophet[datos_prophet['ds'] < fecha])
        pred[fecha] = m.predict(datos_prophet[datos_prophet['ds'] == fecha])['yhat'].iloc[0]
    return a_indice(pd.Series(pred))

In [ ]:
fechas_val_p = pd.date_range('2017-01-01', '2019-12-01', freq='MS')
combinaciones_p = [(cps, sps) for cps in CONFIG['prophet_cps'] for sps in CONFIG['prophet_sps']]
inicio = time.time()
print(f'Prophet - validación: inicio ({len(combinaciones_p)} combinaciones × {len(fechas_val_p)} meses)')
filas = []
for cps, sps in combinaciones_p:
    pred = prophet_a_1_mes(fechas_val_p, changepoint_prior_scale=cps, seasonality_prior_scale=sps)
    filas.append({'changepoint_prior_scale': cps, 'seasonality_prior_scale': sps,
                  'RMSE validación 2017-2019': np.sqrt(mean_squared_error(indice[fechas_val_p], pred))})
print(f'Prophet - validación: fin en {time.time() - inicio:.0f} s')
validacion_prophet = pd.DataFrame(filas)
params_prophet = (validacion_prophet.loc[validacion_prophet['RMSE validación 2017-2019'].idxmin(),
                                         ['changepoint_prior_scale', 'seasonality_prior_scale']].to_dict())
display(validacion_prophet.pivot(index='changepoint_prior_scale', columns='seasonality_prior_scale',
                                 values='RMSE validación 2017-2019')
        .style.format('{:.3f}').highlight_min(color='#c6efce', axis=None)
        .set_caption('Prophet: RMSE del índice a 1 mes en 2017-2019 (verde = combinación elegida)'))
print('Hiperparámetros elegidos:', params_prophet)

In [ ]:
pronosticos_prophet, pronosticos_prophet_h, bandas_prophet, modelos_prophet = {}, {}, {}, {}
inicio = time.time()
print(f'Prophet - test: inicio ({len(regresores_prophet)} versiones × {len(test)} meses)')
for nombre_p, regs in regresores_prophet.items():
    m = crear_prophet(regs, **params_prophet).fit(datos_prophet[datos_prophet['ds'] < inicio_test])
    f = m.predict(datos_prophet[datos_prophet['ds'] >= inicio_test]).set_index('ds')
    modelos_prophet[nombre_p] = m
    pronosticos_prophet_h[nombre_p] = a_indice(f['yhat'])
    bandas_prophet[nombre_p] = a_indice(f[['yhat_lower', 'yhat_upper']])
    pronosticos_prophet[nombre_p] = prophet_a_1_mes(test.index, regs, **params_prophet)
print(f'Prophet - test: fin en {time.time() - inicio:.0f} s')

def fila_prophet(pred_h, pred_1m):
    m_h, m_1 = metricas(indice_test, pred_h), metricas(indice_test, pred_1m)
    return {'RMSE': m_h['RMSE'], 'MAPE (%)': m_h['MAPE (%)'],
            'RMSE 1 mes': m_1['RMSE'], 'MAE 1 mes': m_1['MAE'], 'MAPE (%) 1 mes': m_1['MAPE (%)'],
            'RMSE var. % (pp) 1 mes': np.sqrt(mean_squared_error(variacion_test, a_variacion(pred_1m, indice_original)))}

tabla_prophet = pd.DataFrame({n: fila_prophet(pronosticos_prophet_h[n], pronosticos_prophet[n]) for n in regresores_prophet}).T
tabla_prophet = pd.concat([tabla_prophet, tabla_plana.loc[[etiqueta_sarima, etiqueta_final, etiqueta_arima], tabla_prophet.columns]])
display(tabla_prophet.style.format('{:.4f}').highlight_min(color='#c6efce')
        .set_caption('Prophet frente a SARIMA, SARIMAX y ARIMA (verde = mejor valor de cada columna)'))

coef_prophet = regressor_coefficients(modelos_prophet[nombre_prophet_exog]).set_index('regressor')[['coef']]
coef_prophet.columns = ['Prophet (train)']
coef_prophet['SARIMAX (sección 6.3)'] = [sarimax[nombre_sarimax].params[r] for r in coef_prophet.index]
print('Efecto de los regresores (puntos de 100·ln(índice) por unidad del regresor):')
display(coef_prophet.round(4))

for nombre_p in regresores_prophet:
    dm_p, p_p = diebold_mariano(indice_test, pronosticos_prophet[nombre_p], pronosticos_1m[etiqueta_final])
    print(f'{nombre_p} vs {nombre_sarimax} (1 mes): DM {dm_p:.3f}, p-valor {p_p:.4f} → '
          + ('sin diferencia significativa' if p_p >= alpha else
             f'{nombre_p} es significativamente mejor' if dm_p < 0 else 'el SARIMAX es significativamente mejor'))

In [ ]:
m_p = modelos_prophet[nombre_prophet_exog]
fig = m_p.plot_components(m_p.predict(datos_prophet))
fig.set_size_inches(12, 9)
guardar_figura('ts_07_prophet_componentes.png'); plt.show()

fig, axes = plt.subplots(2, 1, figsize=(14, 9))
axes[0].plot(indice['2018-01-01':], color='black', linewidth=1.4, label='Real')
for nombre_p, pred in pronosticos_prophet_h.items():
    axes[0].plot(pred, linestyle='--', label=nombre_p)
axes[0].fill_between(test.index, bandas_prophet[nombre_prophet_exog].iloc[:, 0],
                     bandas_prophet[nombre_prophet_exog].iloc[:, 1], color='teal', alpha=0.12,
                     label=f'IC 95 % ({nombre_prophet_exog})')
axes[0].plot(pronosticos[etiqueta_final], color='crimson', label=nombre_sarimax)
axes[0].set_title('Prophet: pronóstico de todo el test desde dic-2021 (horizonte 1 a 55 meses)', fontweight='bold')
axes[1].plot(indice['2018-01-01':], color='black', linewidth=1.4, label='Real')
for nombre_p, pred in pronosticos_prophet.items():
    axes[1].plot(pred, linestyle='--', label=nombre_p)
axes[1].plot(pronosticos_1m[etiqueta_final], color='crimson', label=nombre_sarimax)
axes[1].set_title('Prophet: pronóstico a 1 mes, reestimando cada mes', fontweight='bold')
for ax in axes:
    ax.axvspan(test.index[0], test.index[-1], color='lightgray', alpha=0.4)
    ax.grid(alpha=0.3); ax.legend(loc='upper left', fontsize=8)
plt.tight_layout(); guardar_figura('ts_07_prophet_pronostico.png'); plt.show()

## 8. Análisis de residuales del modelo propuesto

In [ ]:
residuales = residuos_validos(modelo)

fig, axes = plt.subplot_mosaic([['t', 't', 'hist'], ['fac', 'facp', 'qq']], figsize=(15, 7))
residuales.plot(ax=axes['t'], title='Residuos en el tiempo')
axes['t'].axhline(0, color='black', linestyle='--')
plot_acf(residuales, lags=36, zero=False, ax=axes['fac'], title='FAC de residuos')
plot_pacf(residuales, lags=36, zero=False, method='ywm', ax=axes['facp'], title='FACP de residuos')
axes['hist'].hist(residuales, bins=30, edgecolor='black')
axes['hist'].set_title('Histograma de residuos')
qqplot(residuales, line='s', ax=axes['qq'])
axes['qq'].set_title('Q–Q de residuos')
plt.tight_layout(); plt.show()

In [ ]:
lb = acorr_ljungbox(residuales, lags=[12, 24], model_df=k)
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    arch = het_arch(residuales, nlags=12)
jb = jarque_bera(residuales)

pruebas = pd.DataFrame({
    'Prueba': ['Ljung–Box (12)', 'Ljung–Box (24)', 'ARCH', 'Jarque–Bera'],
    'Estadístico': [lb['lb_stat'].iloc[0], lb['lb_stat'].iloc[1], arch[0], jb[0]],
    'p-valor': [lb['lb_pvalue'].iloc[0], lb['lb_pvalue'].iloc[1], arch[1], jb[1]],
})
pruebas['Interpretación'] = np.where(pruebas['p-valor'] >= alpha, 'Aceptable: p-valor ≥ 0,05', 'Revisar: p-valor < 0,05')
display(pruebas.round(4))
t_media = stats.ttest_1samp(residuales, 0)
print(f'Media de residuos: {residuales.mean():.4f} (prueba t de media cero: p-valor {t_media.pvalue:.4f})')
print('¿Residuos no correlacionados?', 'Sí → el modelo es adecuado para pronosticar' if lb['lb_pvalue'].min() >= alpha
      else 'No → se vuelve a la identificación del modelo (sección 6)')

## 9. Pronóstico a futuro (12 meses)

In [ ]:
pasos = 12
fechas_futuras = pd.date_range(serie.index[-1] + pd.offsets.MonthBegin(1), periods=pasos, freq='MS')
modelo_extension = {'actividad': (1, (0, 1, 1, 12)), 'precio': (1, (0, 0, 0, 0)), 'clima': (0, (0, 0, 0, 0))}

def extender(x, meses, tipo):
    x = x[inicio_muestra - pd.DateOffset(years=1):FIN].dropna()
    if meses <= 0:
        return x, None
    d_x, estacional_x = modelo_extension[tipo]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        candidatos_x = [ARIMA(x, order=(p, d_x, q), seasonal_order=estacional_x, trend='c' if d_x == 0 else 'n').fit()
                        for p in range(3) for q in range(3) if p + q > 0]
    mejor = min(candidatos_x, key=lambda m: m.bic)
    return pd.concat([x, mejor.get_forecast(steps=meses).predicted_mean]), nombre_modelo(mejor.model.order, mejor.model.seasonal_order)

X_futuro = pd.DataFrame(index=fechas_futuras)
origen = pd.DataFrame(index=fechas_futuras)
for col in cols_exog:
    clave, L = rezago_exog[col]
    tipo = fichas_exogenas.loc[clave, 'tipo']
    if tipo == 'calendario':
        X_futuro[col] = calendario(fechas_futuras)[clave].values
        origen[col] = 'Calendario'
        continue
    extendida, modelo_x = extender(candidatas[clave], pasos - L, tipo)
    X_futuro[col] = extendida.shift(L, freq='MS').reindex(fechas_futuras)
    origen[col] = ['Observado' if i < L else f'Pronosticado {modelo_x}' for i in range(pasos)]
display(pd.concat({'Valor': X_futuro.round(2), 'Origen': origen}, axis=1))

In [ ]:
modelo_completo = ajustar(serie, orden, estacional, exog=X.loc[serie.index, cols_exog])
f_futuro = modelo_completo.get_forecast(steps=pasos, exog=X_futuro[cols_exog])
indice_futuro = a_indice(f_futuro.predicted_mean)
ic_futuro = {nivel: a_indice(f_futuro.conf_int(alpha=1 - nivel / 100)) for nivel in (80, 95)}
base = indice.reindex(fechas_futuras - pd.DateOffset(years=1)).values
sin_exog = a_indice(ajustar(serie, orden, estacional).get_forecast(steps=pasos).predicted_mean)

tabla_futuro = pd.DataFrame({
    'Índice pronosticado': indice_futuro,
    'Var. % interanual': 100 * (indice_futuro / base - 1),
    'Var. % IC 95 % inf.': 100 * (ic_futuro[95].iloc[:, 0].values / base - 1),
    'Var. % IC 95 % sup.': 100 * (ic_futuro[95].iloc[:, 1].values / base - 1),
    'Var. % SARIMA sin exógenas': 100 * (sin_exog.values / base - 1),
}, index=fechas_futuras)
display(tabla_futuro.round(2))
print('Coeficientes reestimados con toda la serie:')
display(pd.DataFrame({'Coeficiente': modelo_completo.params, 'p-valor': modelo_completo.pvalues}).round(4))

fig, axes = plt.subplots(2, 1, figsize=(14, 9))
axes[0].plot(indice['2022-01-01':], color='black', linewidth=1, label='Real')
axes[0].plot(indice_futuro, color='red', linestyle='--', linewidth=1.5, label=f'Pronóstico {nombre_sarimax}')
axes[0].plot(sin_exog, color='green', linestyle=':', linewidth=1.5, label='SARIMA sin exógenas')
for nivel, transparencia in [(95, 0.12), (80, 0.25)]:
    axes[0].fill_between(fechas_futuras, ic_futuro[nivel].iloc[:, 0], ic_futuro[nivel].iloc[:, 1],
                         color='red', alpha=transparencia, linewidth=0, label=f'IC {nivel} %')
axes[0].set_title('Pronóstico a 12 meses - Índice PBI Electricidad y Agua (2007 = 100)', fontweight='bold')
axes[1].plot(variacion_real['2022-01-01':], color='black', linewidth=1, label='Real')
axes[1].plot(tabla_futuro['Var. % interanual'], color='red', linestyle='--', linewidth=1.5, label=f'Pronóstico {nombre_sarimax}')
axes[1].plot(tabla_futuro['Var. % SARIMA sin exógenas'], color='green', linestyle=':', linewidth=1.5, label='SARIMA sin exógenas')
axes[1].fill_between(fechas_futuras, tabla_futuro['Var. % IC 95 % inf.'], tabla_futuro['Var. % IC 95 % sup.'],
                     color='red', alpha=0.12, linewidth=0, label='IC 95 %')
axes[1].axhline(0, color='gray', linewidth=0.6)
axes[1].set_title('Pronóstico a 12 meses - Variación % interanual', fontweight='bold')
for ax in axes:
    ax.grid(alpha=0.3); ax.legend(loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

## 10. Modelos de Machine Learning: XGBoost, LightGBM y CatBoost

### 10.1. Variables del notebook 01

In [ ]:
serie_tiempo = indice.to_frame('Close')

serie_tiempo['Retorno'] = serie_tiempo['Close'].pct_change()
serie_tiempo['Log_Return'] = np.log(serie_tiempo['Close'] / serie_tiempo['Close'].shift(1))
serie_tiempo['Cambio'] = serie_tiempo['Close'].diff()
serie_tiempo['MA_3'] = serie_tiempo['Close'].rolling(3).mean()
serie_tiempo['MA_6'] = serie_tiempo['Close'].rolling(6).mean()
serie_tiempo['MA_9'] = serie_tiempo['Close'].rolling(9).mean()
display(serie_tiempo.tail().round(4))

en_train = serie_tiempo[serie_tiempo.index < inicio_test]
en_test = serie_tiempo[serie_tiempo.index >= inicio_test]
rango = pd.DataFrame({'Train mín.': en_train.min(), 'Train máx.': en_train.max(),
                      'Test mín.': en_test.min(), 'Test máx.': en_test.max(),
                      '% del test por encima del máx. de train': 100 * (en_test > en_train.max()).mean()})
display(rango.round(4))

In [ ]:
def make_lags(df, columna, n_lags):
    data = df.copy()
    for i in range(1, n_lags + 1):
        data[f'lag_{i}'] = df[columna].shift(i)
    return data

variables_ml = pd.DataFrame({'Log_Return': serie_tiempo['Log_Return']})
variables_ml = make_lags(variables_ml, 'Log_Return', 12)
variables_ml['Retorno_t-1'] = serie_tiempo['Retorno'].shift(1)
variables_ml['Cambio_t-1'] = serie_tiempo['Cambio'].shift(1)
for k in (3, 6, 9):
    variables_ml[f'MA_{k}_rel_t-1'] = np.log(serie_tiempo['Close'] / serie_tiempo[f'MA_{k}']).shift(1)
variables_ml['mes'] = variables_ml.index.month
for col in cols_exog:
    clave, L = rezago_exog[col]
    if fichas_exogenas.loc[clave, 'Transformación'] == '100·ln':
        variables_ml[f'ret_{col}'] = (candidatas[clave].diff() / 100).shift(L).reindex(variables_ml.index)
    else:
        variables_ml[col] = candidatas[clave].shift(L).reindex(variables_ml.index)
variables_ml = variables_ml.dropna()

X_ml, y_ml = variables_ml.drop(columns='Log_Return'), variables_ml['Log_Return']
X_train_ml, y_train_ml = X_ml[X_ml.index < inicio_test], y_ml[y_ml.index < inicio_test]
X_test_ml, y_test_ml = X_ml[X_ml.index >= inicio_test], y_ml[y_ml.index >= inicio_test]

print('Predictores:', X_ml.shape[1], '|', ', '.join(X_ml.columns))
print('Train:', X_train_ml.index.min().strftime('%Y-%m'), 'a', X_train_ml.index.max().strftime('%Y-%m'), f'({len(X_train_ml)} meses)')
print('Test: ', X_test_ml.index.min().strftime('%Y-%m'), 'a', X_test_ml.index.max().strftime('%Y-%m'), f'({len(X_test_ml)} meses)')
variables_ml.head(3).round(4)

### 10.2. Entrenamiento y ajuste de hiperparámetros

In [ ]:
cv_temporal = TimeSeriesSplit(n_splits=CONFIG['boost_folds'])
grilla = {'n_estimators': CONFIG['boost_arboles'], 'max_depth': CONFIG['boost_profundidad'], 'learning_rate': CONFIG['boost_tasa']}
grilla_catb = {'iterations': CONFIG['boost_arboles'], 'depth': CONFIG['boost_profundidad'], 'learning_rate': CONFIG['boost_tasa']}
estimadores = {
    'XGBoost': (XGBRegressor(random_state=0, n_jobs=1), grilla),
    'LightGBM': (LGBMRegressor(random_state=0, verbose=-1, n_jobs=1), grilla),
    'CatBoost': (CatBoostRegressor(random_seed=0, logging_level='Silent', allow_writing_files=False, thread_count=1), grilla_catb),
}

modelos_ml, mejores_param = {}, []
for nombre, (estimador, grid) in estimadores.items():
    busqueda_ml = GridSearchCV(estimador, grid, cv=cv_temporal, scoring='neg_root_mean_squared_error', refit=True, n_jobs=-1)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        busqueda_ml.fit(X_train_ml, y_train_ml)
    modelos_ml[nombre] = busqueda_ml.best_estimator_
    p = busqueda_ml.best_params_
    mejores_param.append({'Modelo': nombre, 'Árboles': p.get('n_estimators', p.get('iterations')),
                          'Profundidad': p.get('max_depth', p.get('depth')), 'Tasa de aprendizaje': p['learning_rate'],
                          'RMSE validación (Log_Return)': -busqueda_ml.best_score_})
display(pd.DataFrame(mejores_param).set_index('Modelo').round(5))

In [ ]:
importancias = pd.DataFrame({nombre: m.feature_importances_ / m.feature_importances_.sum()
                             for nombre, m in modelos_ml.items()}, index=X_train_ml.columns)
importancias['Promedio'] = importancias.mean(axis=1)
importancias = importancias.sort_values('Promedio', ascending=False)
display(importancias.head(10).round(3))

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for ax, nombre in zip(axes, modelos_ml):
    ax.barh(importancias.index, importancias[nombre], color='steelblue')
    ax.set_title(f'Importancia relativa - {nombre}', fontweight='bold'); ax.grid(alpha=0.3, axis='x')
axes[0].invert_yaxis()
plt.tight_layout(); plt.show()

### 10.3. Pronóstico a 1 mes y métricas

In [ ]:
def a_indice_ml(pred_log_return):
    return pd.Series(indice.shift(1)[X_test_ml.index].values * np.exp(pred_log_return), index=X_test_ml.index)

def IEC(mse, rmse, mae, mape):
    return np.mean([mse / (1 + mse), rmse / (1 + rmse), mae / (1 + mae), mape / (1 + mape)])

pronosticos_ml = {nombre: a_indice_ml(m.predict(X_test_ml)) for nombre, m in modelos_ml.items()}
pronosticos_ml['Ensamble'] = (pronosticos_ml['XGBoost'] + pronosticos_ml['LightGBM'] + pronosticos_ml['CatBoost']) / 3

def tabla_metricas(pronos):
    filas = []
    for nombre, pred in pronos.items():
        fila = {'Modelo': nombre, **metricas(indice_test, pred)}
        fila['RMSE var. % (pp)'] = np.sqrt(mean_squared_error(variacion_test, a_variacion(pred, indice_original)))
        fila['IEC'] = IEC(fila['MSE'], fila['RMSE'], fila['MAE'], fila['MAPE (%)'])
        filas.append(fila)
    return pd.DataFrame(filas).set_index('Modelo')

metricas_ml = tabla_metricas(pronosticos_ml)
display(metricas_ml.round(4))

## 11. AutoML con H2O

In [ ]:
import h2o
from h2o.automl import H2OAutoML

h2o.init(verbose=False)
h2o.no_progress()

datos_train_h2o = pd.concat([X_train_ml, y_train_ml], axis=1)
datos_train_h2o['bloque'] = np.arange(len(datos_train_h2o)) * 5 // len(datos_train_h2o)
print('Meses por bloque de validación:', datos_train_h2o['bloque'].value_counts().sort_index().tolist())
train_h2o = h2o.H2OFrame(datos_train_h2o.reset_index(drop=True))
test_h2o = h2o.H2OFrame(X_test_ml.reset_index(drop=True))

automl_model = H2OAutoML(max_models=CONFIG['automl_max_modelos'], seed=1, sort_metric='RMSE', max_runtime_secs=CONFIG['automl_segundos'])
automl_model.train(x=list(X_train_ml.columns), y='Log_Return', training_frame=train_h2o, fold_column='bloque')

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    lb = automl_model.leaderboard.as_data_frame()
lb[['model_id', 'rmse', 'mae']]

In [ ]:
automl_model_best = automl_model.leader
print('Mejor modelo de AutoML:', automl_model_best.model_id)

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    pred_automl = automl_model_best.predict(test_h2o).as_data_frame()['predict'].values
pronosticos_ml['AutoML (H2O)'] = a_indice_ml(pred_automl)
display(tabla_metricas({'AutoML (H2O)': pronosticos_ml['AutoML (H2O)']}).round(4))

## 12. Comparación final: SARIMAX, boosting y AutoML

In [ ]:
comparacion_final = tabla_metricas({
    nombre_sarimax: pronosticos_1m[etiqueta_final],
    'SARIMA': pronosticos_1m[etiqueta_sarima],
    **pronosticos_ml,
    'Ingenuo con último crecimiento': pronosticos_1m['Ingenuo con último crecimiento'],
    'Ingenuo estacional': pronosticos_1m['Ingenuo estacional'],
}).sort_values('RMSE')
comparacion_final.insert(0, 'Tipo', ['Econometría' if 'SARIMA' in m else 'Referencia' if 'Ingenuo' in m
                                     else 'AutoML' if 'AutoML' in m else 'Boosting' for m in comparacion_final.index])
display(comparacion_final.style.format('{:.4f}', subset=comparacion_final.columns[1:])
        .highlight_min(subset=comparacion_final.columns[1:], color='#c6efce')
        .set_caption('Test a 1 mes, métricas sobre el índice (verde = mejor valor de cada columna)'))

colores_tipo = {'Econometría': 'crimson', 'Boosting': 'steelblue', 'AutoML': 'darkorange', 'Referencia': 'gray'}
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
for ax, columna in zip(axes, ['RMSE', 'MAPE (%)', 'IEC']):
    ax.barh(comparacion_final.index, comparacion_final[columna], color=comparacion_final['Tipo'].map(colores_tipo))
    for i, v in enumerate(comparacion_final[columna]):
        ax.text(v, i, f' {v:.3f}', va='center', fontsize=8)
    ax.set_title(columna, fontweight='bold'); ax.grid(alpha=0.3, axis='x')
axes[0].invert_yaxis()
plt.suptitle('Test a 1 mes (rojo = econometría, azul = boosting, naranja = AutoML, gris = ingenuos)', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
mejor_boosting = metricas_ml.drop(index='Ensamble')['RMSE'].idxmin()
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(indice['2021-01-01':], color='black', linewidth=1.5, label='Real')
for nombre, serie_pred, estilo in [(nombre_sarimax, pronosticos_1m[etiqueta_final], '-'),
                                   (mejor_boosting, pronosticos_ml[mejor_boosting], '--'),
                                   ('Ensamble boosting', pronosticos_ml['Ensamble'], '--'),
                                   ('AutoML (H2O)', pronosticos_ml['AutoML (H2O)'], ':')]:
    ax.plot(serie_pred, linestyle=estilo, linewidth=1.2, label=nombre)
ax.axvspan(test.index[0], test.index[-1], color='lightgray', alpha=0.4)
ax.set_title('Índice PBI Electricidad y Agua: pronóstico a 1 mes en el test', fontweight='bold')
ax.grid(alpha=0.3); ax.legend(loc='upper left', fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
pares = [(nombre_sarimax, 'AutoML (H2O)'), (nombre_sarimax, 'Ensamble'),
         (nombre_sarimax, mejor_boosting), ('AutoML (H2O)', 'Ensamble')]
pares += [('AutoML (H2O)', nombre) for nombre in ['XGBoost', 'LightGBM', 'CatBoost']]
todos = {nombre_sarimax: pronosticos_1m[etiqueta_final], **pronosticos_ml}

dm_ml = []
for a, b in pares:
    dm, p = diebold_mariano(indice_test, todos[a], todos[b])
    dm_ml.append({'Comparación': f'{a} vs {b}', 'Estadístico DM': dm, 'p-valor': p,
                  'Conclusión': 'Sin diferencia significativa' if p >= alpha else
                                f'{a} es significativamente mejor' if dm < 0 else f'{b} es significativamente mejor'})
display(pd.DataFrame(dm_ml).drop_duplicates('Comparación').round(4))

In [ ]:
errores_abs = pd.DataFrame({nombre: (indice_test - pred).abs() for nombre, pred in {
    nombre_sarimax: pronosticos_1m[etiqueta_final], 'SARIMA': pronosticos_1m[etiqueta_sarima],
    **pronosticos_ml, 'Ingenuo con último crecimiento': pronosticos_1m['Ingenuo con último crecimiento']}.items()})
deciles_error = errores_abs.quantile(np.arange(0.1, 1.0, 0.1)).T
deciles_error.columns = [f'D{i}' for i in range(1, 10)]
deciles_error['Máximo'] = errores_abs.max()
deciles_error = deciles_error.sort_values('D5')
print('Deciles del error absoluto a 1 mes (puntos del índice):')
display(deciles_error.style.format('{:.2f}').highlight_min(color='#c6efce'))

plt.figure(figsize=(14, 4.5))
plt.boxplot([errores_abs[m] for m in deciles_error.index], widths=0.6)
plt.xticks(range(1, len(deciles_error) + 1), [m.replace(' + ', '\n+ ').replace(' con ', '\ncon ') for m in deciles_error.index], fontsize=8)
plt.ylabel('|real − pronóstico| (puntos)'); plt.grid(alpha=0.3)
plt.title('Distribución del error absoluto a 1 mes en el test (cuartiles)', fontweight='bold')
plt.tight_layout(); plt.show()

## 13. Detección de anomalías con Isolation Forest (H2O)

### 13.1. Preparación de los datos

In [ ]:
dias = pd.Series(indice_original.index.days_in_month, index=indice_original.index)
dias_previo = pd.Series((indice_original.index - pd.DateOffset(years=1)).days_in_month, index=indice_original.index)

def variables_if(ind):
    var = (100 * (ind / ind.shift(12) - 1)).fillna(variacion_muestra)
    d = pd.DataFrame({'var_interanual_dia': 100 * ((1 + var / 100) * dias_previo / dias - 1),
                      'crec_mensual_dia': 100 * np.log(ind / dias).diff()})
    d['mes'] = d.index.month
    d['crec_mensual_desest'] = d['crec_mensual_dia'] - d.groupby('mes')['crec_mensual_dia'].transform('median')
    return d.drop(columns='crec_mensual_dia').dropna()

variables = ['var_interanual_dia', 'crec_mensual_desest']
datos_if = variables_if(indice_original)
datos_if['pandemia'] = datos_if.index.isin(pandemia)

print('Periodo:', datos_if.index.min().strftime('%Y-%m'), 'a', datos_if.index.max().strftime('%Y-%m'), f'({len(datos_if)} meses)')
display(datos_if.head().round(2))
display(datos_if[variables].describe().T.round(2))

### 13.2. Modelado

In [ ]:
import h2o
from h2o.estimators import H2OIsolationForestEstimator

h2o.init(verbose=False)
h2o.no_progress()

def a_pandas(marco):
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return marco.as_data_frame()

data_if = h2o.H2OFrame(datos_if[variables].reset_index(drop=True))
modelo_if = H2OIsolationForestEstimator(ntrees=CONFIG['if_arboles'],
                                        sample_rate=0.1,
                                        seed=1234)
modelo_if.train(x=variables, training_frame=data_if)

predicciones = a_pandas(modelo_if.predict(data_if))
predicciones.index = datos_if.index
display(predicciones.head().round(3))
print('Correlación entre predict y mean_length:', round(predicciones['predict'].corr(predicciones['mean_length']), 3))

### 13.3. Estabilidad entre semillas

In [ ]:
from itertools import combinations
from scipy.stats import spearmanr

def puntuar_if(datos, columnas, semillas):
    marco = h2o.H2OFrame(datos[columnas].reset_index(drop=True))
    scores = {}
    for semilla in semillas:
        m = H2OIsolationForestEstimator(ntrees=CONFIG['if_arboles'], sample_rate=0.1, seed=semilla)
        m.train(x=columnas, training_frame=marco)
        scores[f'semilla {semilla}'] = a_pandas(m.predict(marco))['predict'].values
    return pd.DataFrame(scores, index=datos.index)

def top_5(score):
    return set(score.index[score > score.quantile(0.95)])

def jaccard(a, b):
    return len(a & b) / len(a | b)

SEMILLAS_IF = range(1, CONFIG['if_semillas'] + 1)
scores = puntuar_if(datos_if, variables, SEMILLAS_IF)
score_prom = scores.mean(axis=1).rename('score')

pares_semillas = list(combinations(scores.columns, 2))
mitad = scores.shape[1] // 2
mitad_1, mitad_2 = scores.iloc[:, :mitad].mean(axis=1), scores.iloc[:, mitad:].mean(axis=1)
estabilidad = pd.Series({
    'Spearman entre semillas (promedio de los pares)': np.mean([spearmanr(scores[a], scores[b])[0] for a, b in pares_semillas]),
    'Jaccard del top 5 % entre semillas (promedio de los pares)': np.mean([jaccard(top_5(scores[a]), top_5(scores[b])) for a, b in pares_semillas]),
    f'Jaccard del top 5 % entre dos promedios de {mitad} semillas': jaccard(top_5(mitad_1), top_5(mitad_2)),
}, name='Valor')
display(estabilidad.to_frame().round(2))

frecuencia = pd.Series({t: sum(t in top_5(scores[col]) for col in scores.columns) for t in scores.index})
print(f'Meses que al menos una semilla señala en su top 5 %: {int((frecuencia > 0).sum())} | '
      f'meses que señalan las {len(SEMILLAS_IF)} semillas: {int((frecuencia == len(SEMILLAS_IF)).sum())} | máximo de semillas que coinciden en un mes: {int(frecuencia.max())}')

### 13.4. Meses señalados como anomalías

In [ ]:
datos_if['score'] = score_prom
umbral = score_prom.quantile(0.95)
alertas = datos_if[datos_if['score'] > umbral].sort_values('score', ascending=False)
print(f'Umbral (percentil 95 del score promedio): {umbral:.3f} | meses señalados: {len(alertas)} de {len(datos_if)}')

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
axes[0].plot(indice_original, color='gray', linewidth=1, label='Índice real (sin imputar)')
axes[0].scatter(alertas.index, indice_original[alertas.index], color='red', zorder=3, label='Anomalías (top 5 % del score)')
axes[0].axvspan(pandemia[0], pandemia[-1], color='orange', alpha=0.15, label='Meses imputados en 3.1 (pandemia)')
axes[0].set_title('Detección de anomalías con Isolation Forest: PBI Electricidad y Agua', fontweight='bold')
axes[0].legend(loc='upper left', fontsize=8)
axes[1].bar(datos_if.index, datos_if['score'], width=25, color=np.where(datos_if['score'] > umbral, 'red', 'steelblue'))
axes[1].axhline(umbral, color='red', linestyle=':', linewidth=1, label='Percentil 95')
axes[1].axvspan(pandemia[0], pandemia[-1], color='orange', alpha=0.15)
axes[1].set_title(f'Score de anomalía (promedio de {len(SEMILLAS_IF)} semillas)', fontweight='bold')
axes[1].legend(loc='upper left', fontsize=8)
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

tabla_alertas = pd.DataFrame({
    'Score': alertas['score'],
    'Índice': indice_original[alertas.index],
    'Var. % interanual (BCRP)': variacion_muestra[alertas.index],
    'Var. % interanual por día': alertas['var_interanual_dia'],
    'Crec. mensual desest. (%)': alertas['crec_mensual_desest'],
    'Pandemia (3.1)': np.where(alertas['pandemia'], 'Sí', 'No'),
    'Atípico 1,5·IQR (3.4)': np.where(alertas.index.isin(atipicos_var.index), 'Sí', 'No'),
})
tabla_alertas.index = tabla_alertas.index.strftime('%Y-%m')
display(tabla_alertas.round(2))

### 13.5. Deciles del score

In [ ]:
datos_if['decil_score'] = pd.qcut(datos_if['score'].rank(method='first'), 10, labels=range(1, 11)).astype(int)
datos_if['atipico_iqr'] = datos_if.index.isin(atipicos_var.index)

por_decil = datos_if.groupby('decil_score').agg(**{
    'Meses': ('score', 'size'),
    'Score mínimo': ('score', 'min'),
    'Score máximo': ('score', 'max'),
    'Meses de pandemia': ('pandemia', 'sum'),
    'Atípicos 1,5·IQR (3.4)': ('atipico_iqr', 'sum'),
    'Prom. |var. % interanual por día|': ('var_interanual_dia', lambda x: x.abs().mean()),
})
por_decil.index = [f'D{i}' for i in por_decil.index]
display(por_decil.round(3))

prioridad = datos_if[datos_if['decil_score'] == 10].sort_values('score', ascending=False)
print(f'Casos de revisión prioritaria (decil 10): {len(prioridad)} meses | {int(prioridad["pandemia"].sum())} son de la pandemia y '
      f'{int((~prioridad["pandemia"]).sum())} no')
print('Meses del decil 10 fuera de la pandemia:', ', '.join(prioridad.index[~prioridad['pandemia']].strftime('%Y-%m').sort_values()))

### 13.6. Contraste: nivel del índice, variable de mes y serie imputada

In [ ]:
from sklearn.metrics import roc_auc_score

def evaluar_if(nombre, datos, columnas, semillas=SEMILLAS_IF):
    s = puntuar_if(datos, columnas, semillas).mean(axis=1)
    top = s[s > s.quantile(0.95)].index
    en_pandemia = datos.index.isin(pandemia)
    return {'Modelo': nombre, 'Variables': ', '.join(columnas),
            'Pandemia en el top 5 %': f'{int(top.isin(pandemia).sum())} de {len(top)}',
            'AUC (pandemia)': roc_auc_score(en_pandemia, s),
            'Top 5 % en 2002-2004 o 2024-2026': f'{int(((top.year <= 2004) | (top.year >= 2024)).sum())} de {len(top)}'}

datos_nivel = pd.DataFrame({'Close': indice_original, 'mes': indice_original.index.month})
datos_imputada = variables_if(indice)

contraste = pd.DataFrame([
    evaluar_if('A. Nivel del índice + mes (como en Bitcoin)', datos_nivel, ['Close', 'mes']),
    evaluar_if('B. Variables propuestas, serie real', datos_if, variables),
    evaluar_if('B2. Variables propuestas + mes, serie real', datos_if, variables + ['mes']),
    evaluar_if('C. Variables propuestas, serie con pandemia imputada', datos_imputada, variables),
]).set_index('Modelo')
display(contraste.style.format({'AUC (pandemia)': '{:.3f}'}).set_caption('Las filas A, B y B2 usan la serie real; la fila C, la serie con la pandemia imputada en 3.1'))

### 13.7. Exportación y cierre del clúster H2O

In [ ]:
exportar = prioridad[['score', 'decil_score', 'var_interanual_dia', 'crec_mensual_desest', 'pandemia', 'atipico_iqr']].copy()
exportar.insert(1, 'indice', indice_original[exportar.index])
exportar.index = exportar.index.strftime('%Y-%m').rename('mes_anio')
exportar.to_csv('data_anomalia_electricidad_agua.csv', encoding='utf-8-sig')
print(f'Exportados {len(exportar)} meses del decil 10 a data_anomalia_electricidad_agua.csv')
exportar.head()

In [ ]:
h2o.cluster().shutdown()

## 14. Deep Learning: redes neuronales recurrentes (LSTM, GRU y bidireccionales)

### 14.1. Adaptación del notebook 12 a esta serie

### 14.2. Librerías

In [ ]:
import joblib

gpu_torch = False
if importlib.util.find_spec('torch') is not None:
    import torch
    gpu_torch = torch.cuda.is_available()
if gpu_torch:
    os.environ['KERAS_BACKEND'] = 'torch'
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

import keras
from keras.models import Sequential, load_model
from keras.layers import Input, Dense, LSTM, GRU, Bidirectional
from keras.utils import set_random_seed
from keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler

if keras.backend.backend() == 'torch':
    dispositivo = f'GPU {torch.cuda.get_device_name(0)}' if gpu_torch else 'CPU'
elif keras.backend.backend() == 'tensorflow':
    import tensorflow as tf
    dispositivo = 'GPU' if tf.config.list_physical_devices('GPU') else 'CPU'
else:
    dispositivo = 'CPU'
print('Keras', keras.__version__, '| motor:', keras.backend.backend(), '| dispositivo:', dispositivo)

### 14.3. Preparación de los datos

In [ ]:
VENTANA = 24
EPOCAS = CONFIG['rnn_epocas_max']
PACIENCIA_RNN = CONFIG['rnn_paciencia']
LOTE_RNN = CONFIG['rnn_lote']
N_VAL_RNN = CONFIG['rnn_n_val']

exog_rnn = {}
for col in cols_exog:
    clave, _ = rezago_exog[col]
    ficha = fichas_exogenas.loc[clave]
    if ficha['tipo'] == 'calendario':
        exog_rnn[f'{clave}_t+1'] = pd.Series(calendario(serie_tiempo.index + pd.offsets.MonthBegin(1))[clave].values,
                                             index=serie_tiempo.index)
        continue
    atraso = int(ficha['Rezago mínimo']) - 1
    x = candidatas[clave].diff() / 100 if ficha['Transformación'] == '100·ln' else candidatas[clave]
    nombre_x = f'ret_{clave}' if ficha['Transformación'] == '100·ln' else clave
    exog_rnn[f'{nombre_x}_t-{atraso}' if atraso else nombre_x] = x.shift(atraso)

datos_rnn = pd.DataFrame({'Log_Return': serie_tiempo['Log_Return'], **exog_rnn}).loc[serie_tiempo.index].dropna()
datos_rnn['mes_sin'] = np.sin(2 * np.pi * datos_rnn.index.month / 12)
datos_rnn['mes_cos'] = np.cos(2 * np.pi * datos_rnn.index.month / 12)
en_train_rnn = datos_rnn.index < inicio_test

revision = pd.concat([serie_tiempo[['Close']].loc[datos_rnn.index], datos_rnn[['Log_Return', *exog_rnn]]], axis=1)
rango_rnn = pd.DataFrame({
    'Train mín.': revision[en_train_rnn].min(), 'Train máx.': revision[en_train_rnn].max(),
    'Test mín.': revision[~en_train_rnn].min(), 'Test máx.': revision[~en_train_rnn].max(),
    '% del test por encima del máx. de train': 100 * (revision[~en_train_rnn] > revision[en_train_rnn].max()).mean(),
})
escalado = MinMaxScaler().fit(revision[en_train_rnn]).transform(revision[~en_train_rnn])
rango_rnn['Test máx. escalado (MinMax de train)'] = escalado.max(axis=0)
print('Periodo:', datos_rnn.index.min().strftime('%Y-%m'), 'a', datos_rnn.index.max().strftime('%Y-%m'), f'({len(datos_rnn)} meses)')
display(rango_rnn.round(4))

In [ ]:
def crear_ventanas(columnas, ventana=VENTANA, corte=inicio_test):
    en_train = datos_rnn.index < corte
    escalador_x = MinMaxScaler(feature_range=(0, 1)).fit(datos_rnn.loc[en_train, columnas])
    escalador_y = MinMaxScaler(feature_range=(0, 1)).fit(datos_rnn.loc[en_train, ['Log_Return']])
    x_esc = escalador_x.transform(datos_rnn[columnas])
    y_esc = escalador_y.transform(datos_rnn[['Log_Return']])[:, 0]

    x, y = [], []
    for i in range(ventana, len(datos_rnn)):
        x.append(x_esc[i - ventana:i, :])
        y.append(y_esc[i])
    x, y = np.array(x, dtype='float32'), np.array(y, dtype='float32')
    fechas = datos_rnn.index[ventana:]
    es_train = fechas < corte
    return {'columnas': columnas, 'x_train': x[es_train], 'y_train': y[es_train], 'fechas_train': fechas[es_train],
            'x_test': x[~es_train], 'fechas_test': fechas[~es_train],
            'escalador_x': escalador_x, 'escalador_y': escalador_y}

conjuntos_rnn = {'Sin exógenas': ['Log_Return', 'mes_sin', 'mes_cos'],
                 'Con exógenas': ['Log_Return', *exog_rnn, 'mes_sin', 'mes_cos']}
ventanas_rnn = {nombre: crear_ventanas(columnas) for nombre, columnas in conjuntos_rnn.items()}

for nombre, v in ventanas_rnn.items():
    assert v['fechas_test'].equals(X_test_ml.index)
    print(f"{nombre:<13} x_train {v['x_train'].shape} | y_train {v['y_train'].shape} | x_test {v['x_test'].shape} "
          f"| train {v['fechas_train'][0]:%Y-%m} a {v['fechas_train'][-1]:%Y-%m} | test {v['fechas_test'][0]:%Y-%m} a {v['fechas_test'][-1]:%Y-%m}")

v = ventanas_rnn['Con exógenas']
print(f"\nPrimera muestra de train: objetivo = Log_Return de {v['fechas_train'][0]:%Y-%m} (escalado {v['y_train'][0]:.4f})")
display(pd.DataFrame(v['x_train'][0][-3:], columns=v['columnas'],
                     index=pd.date_range(end=v['fechas_train'][0] - pd.offsets.MonthBegin(1), periods=3, freq='MS').strftime('%Y-%m')).round(4))

### 14.4. Modelado: arquitectura de las redes

In [ ]:
def crear_red(tipo, bidireccional, n_variables, ventana=VENTANA, nombre=None):
    Capa, neuronas = (LSTM, CONFIG['rnn_neuronas_lstm']) if tipo == 'LSTM' else (GRU, CONFIG['rnn_neuronas_gru'])
    envolver = Bidirectional if bidireccional else (lambda capa: capa)
    red = Sequential(name=nombre)
    red.add(Input(shape=(ventana, n_variables)))
    red.add(envolver(Capa(neuronas, return_sequences=True, activation='relu')))
    red.add(envolver(Capa(CONFIG['rnn_neuronas_capa2'], return_sequences=False)))
    red.add(Dense(CONFIG['rnn_densa']))
    red.add(Dense(1))
    red.compile(optimizer='adam', loss='mean_squared_error')
    return red


def entrenar_rnn(v, tipo, bidireccional, semilla=0):
    x, y = v['x_train'], v['y_train']
    set_random_seed(semilla)
    red = crear_red(tipo, bidireccional, x.shape[2], x.shape[1])
    parada = EarlyStopping(monitor='val_loss', patience=PACIENCIA_RNN, restore_best_weights=True)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        historia = red.fit(x[:-N_VAL_RNN], y[:-N_VAL_RNN], validation_data=(x[-N_VAL_RNN:], y[-N_VAL_RNN:]),
                           batch_size=LOTE_RNN, epochs=EPOCAS, callbacks=[parada], verbose=0).history
    mejor_epoca = int(np.argmin(historia['val_loss'])) + 1
    set_random_seed(semilla)
    red = crear_red(tipo, bidireccional, x.shape[2], x.shape[1])
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        red.fit(x, y, batch_size=LOTE_RNN, epochs=mejor_epoca, verbose=0)
    return red, mejor_epoca, historia

n_sin, n_con = len(conjuntos_rnn['Sin exógenas']), len(conjuntos_rnn['Con exógenas'])
arquitecturas = {'LSTM': ('LSTM', False), 'GRU': ('GRU', False),
                 'LSTM bidireccional': ('LSTM', True), 'GRU bidireccional': ('GRU', True)}

filas = []
for nombre, (tipo, bid) in arquitecturas.items():
    neuronas, copias = (CONFIG['rnn_neuronas_lstm'] if tipo == 'LSTM' else CONFIG['rnn_neuronas_gru']), (2 if bid else 1)
    capa1 = f'{tipo}({neuronas}, relu)'
    capa2 = f"{tipo}({CONFIG['rnn_neuronas_capa2']})"
    filas.append({'Red': nombre,
                  'Capa 1': f'Bidirectional({capa1})' if bid else capa1, 'Capa 2': f'Bidirectional({capa2})' if bid else capa2,
                  'Salida capa 1': f'({VENTANA}, {neuronas * copias})', 'Salida capa 2': f"({CONFIG['rnn_neuronas_capa2'] * copias},)",
                  f'Parámetros ({n_sin} variables)': crear_red(tipo, bid, n_sin).count_params(),
                  f'Parámetros ({n_con} variables)': crear_red(tipo, bid, n_con).count_params()})
arquitectura_rnn = pd.DataFrame(filas).set_index('Red')
display(arquitectura_rnn)
print(f"Muestras de entrenamiento: {len(ventanas_rnn['Con exógenas']['y_train'])} "
      f"(el notebook 12 tenía 3 901 muestras para 117 619 parámetros)")
crear_red('LSTM', True, n_con, nombre='LSTM_bidireccional').summary()

### 14.5. Elección de las variables con una validación dentro de train

In [ ]:
VALIDAR = CONFIG['rnn_validar']
corte_val, fin_val = pd.Timestamp('2017-01-01'), pd.Timestamp('2019-12-01')

def rmse_validacion(columnas, ventana, tipo, bidireccional, semilla=0):
    v = crear_ventanas(columnas, ventana, corte=corte_val)
    evaluar = v['fechas_test'] <= fin_val
    red, _, _ = entrenar_rnn(v, tipo, bidireccional, semilla)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        pred = v['escalador_y'].inverse_transform(red.predict(v['x_test'][evaluar], verbose=0))[:, 0]
    fechas = v['fechas_test'][evaluar]
    return np.sqrt(mean_squared_error(indice[fechas], indice.shift(1)[fechas].values * np.exp(pred)))

if VALIDAR:
    disenos = {'Univariada': ['Log_Return'], 'Univariada + mes': conjuntos_rnn['Sin exógenas'],
               'Con exógenas': [col for col in conjuntos_rnn['Con exógenas'] if not col.startswith('mes_')], 'Con exógenas + mes': conjuntos_rnn['Con exógenas']}
    tabla_diseno = pd.DataFrame({d: {n: rmse_validacion(c, VENTANA, *a) for n, a in arquitecturas.items()} for d, c in disenos.items()})
    print('RMSE de validación según las variables (ventana de 24 meses):'); display(tabla_diseno.round(3))
else:
    print('Validación no ejecutada (VALIDAR = False): sus resultados están en la tabla de arriba.')

### 14.6. Entrenamiento con parada temprana

In [ ]:
redes_rnn, historial_rnn, pronosticos_rnn, tiempos_rnn, epocas_rnn = {}, {}, {}, {}, {}
for conjunto, v in ventanas_rnn.items():
    for nombre, (tipo, bid) in arquitecturas.items():
        inicio = time.time()
        red, epocas, historia = entrenar_rnn(v, tipo, bid, semilla=0)
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            prediccion = v['escalador_y'].inverse_transform(red.predict(v['x_test'], verbose=0))[:, 0]
        redes_rnn[(conjunto, nombre)] = red
        historial_rnn[(conjunto, nombre)] = historia['val_loss']
        epocas_rnn[(conjunto, nombre)] = epocas
        tiempos_rnn[(conjunto, nombre)] = time.time() - inicio
        pronosticos_rnn[(conjunto, nombre)] = a_indice_ml(prediccion)
        print(f'{conjunto:<13} {nombre:<19} {tiempos_rnn[(conjunto, nombre)]:5.1f} s en {dispositivo} | {epocas:3d} épocas | pérdida de validación mínima: {min(historia["val_loss"]):.5f}')

tabla_epocas = pd.DataFrame({clave: pd.Series(h, index=range(1, len(h) + 1)) for clave, h in historial_rnn.items()})
tabla_epocas.index.name = 'Época'
display(tabla_epocas.style.format('{:.5f}').highlight_min(color='#c6efce')
        .set_caption(f'Pérdida de validación interna ({N_VAL_RNN} meses) por época (MSE del Log_Return escalado)'))

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, conjunto in zip(axes, ventanas_rnn):
    for nombre in arquitecturas:
        ax.plot(tabla_epocas.index, tabla_epocas[(conjunto, nombre)], label=f'{nombre} ({epocas_rnn[(conjunto, nombre)]} épocas)')
    ax.set_title(f'Pérdida de validación por época - {conjunto.lower()}', fontweight='bold'); ax.set_xlabel('Época')
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
axes[0].set_ylabel('MSE (escalado)')
plt.tight_layout(); plt.show()

### 14.7. Evaluación: tabla de métricas de LSTM, GRU y bidireccionales

In [ ]:
metricas_rnn = pd.concat({conjunto: tabla_metricas({nombre: pronosticos_rnn[(conjunto, nombre)] for nombre in arquitecturas})
                          for conjunto in ventanas_rnn}, names=['Variables', 'Modelo'])
display(metricas_rnn.style.format('{:.4f}').highlight_min(color='#c6efce')
        .set_caption(f'LSTM, GRU y bidireccionales (hasta {EPOCAS} épocas, parada temprana) - test a 1 mes, métricas sobre el índice (verde = mejor valor)'))

for conjunto in ventanas_rnn:
    metricas_df = metricas_rnn.loc[conjunto, ['MSE', 'RMSE', 'MAE', 'MAPE (%)']].reset_index()
    print(f'\n{conjunto} ({len(conjuntos_rnn[conjunto])} variables por mes):')
    display(metricas_df.round(4))

print('\nReferencia (sección 12):')
display(comparacion_final.loc[[nombre_sarimax, 'SARIMA', 'CatBoost', 'Ingenuo con último crecimiento'],
                              ['MSE', 'RMSE', 'MAE', 'MAPE (%)']].round(4))

### 14.8. Visualización de las predicciones

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 10))
for ax, conjunto in zip(axes, ventanas_rnn):
    ax.plot(indice['2019-01-01':inicio_test - pd.DateOffset(months=1)], color='gray', linewidth=1.2, label='Train')
    ax.plot(indice_test, color='black', linewidth=1.6, label='Real (test)')
    for nombre in arquitecturas:
        ax.plot(pronosticos_rnn[(conjunto, nombre)], linestyle='--', linewidth=1.1, label=nombre)
    ax.plot(pronosticos_1m[etiqueta_final], color='crimson', linewidth=1.3, label=nombre_sarimax)
    ax.axvspan(test.index[0], test.index[-1], color='lightgray', alpha=0.3)
    ax.set_title(f'Índice PBI Electricidad y Agua: pronóstico a 1 mes - redes {conjunto.lower()}', fontweight='bold')
    ax.set_ylabel('Índice (2007 = 100)'); ax.grid(alpha=0.3); ax.legend(loc='upper left', fontsize=8, ncol=2)
plt.tight_layout(); plt.show()

mejor_conjunto = metricas_rnn.groupby(level='Variables')['RMSE'].min().idxmin()
valid = pd.concat({'Real': indice_test, **{nombre: pronosticos_rnn[(mejor_conjunto, nombre)] for nombre in arquitecturas},
                   nombre_sarimax: pronosticos_1m[etiqueta_final]}, axis=1)
valid.index = valid.index.strftime('%Y-%m')
print(f'Real vs pronosticado (redes {mejor_conjunto.lower()}), últimos 12 meses del test:')
display(valid.tail(12).round(2))

### 14.9. ¿Los resultados dependen de la semilla?

In [ ]:
SEMILLAS = range(CONFIG['rnn_semillas'])
conjunto_semillas = 'Sin exógenas'
v = ventanas_rnn[conjunto_semillas]
filas = []
for nombre, (tipo, bid) in arquitecturas.items():
    for semilla in SEMILLAS:
        if semilla == 0:
            pred = pronosticos_rnn[(conjunto_semillas, nombre)]
        else:
            red, _, _ = entrenar_rnn(v, tipo, bid, semilla)
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                pred = a_indice_ml(v['escalador_y'].inverse_transform(red.predict(v['x_test'], verbose=0))[:, 0])
        filas.append({'Modelo': nombre, 'Semilla': semilla, 'RMSE': np.sqrt(mean_squared_error(indice_test, pred)),
                      'MAPE (%)': metricas(indice_test, pred)['MAPE (%)'], 'pred': pred})
semillas_rnn = pd.DataFrame(filas)

rmse_sarimax = comparacion_final.loc[nombre_sarimax, 'RMSE']
por_red = semillas_rnn.groupby('Modelo', sort=False)
resumen_semillas = por_red['RMSE'].agg(['mean', 'std', 'min', 'max'])
resumen_semillas.columns = ['RMSE promedio', 'Desv. estándar', 'RMSE mínimo', 'RMSE máximo']
resumen_semillas['MAPE promedio (%)'] = por_red['MAPE (%)'].mean()
resumen_semillas['Semillas que superan al SARIMAX'] = por_red['RMSE'].apply(lambda r: f'{(r < rmse_sarimax).sum()} de {len(r)}')
display(resumen_semillas.style.format('{:.4f}', subset=resumen_semillas.columns[:-1])
        .set_caption(f'RMSE de test con {len(SEMILLAS)} semillas - redes {conjunto_semillas.lower()} (SARIMAX: {rmse_sarimax:.4f})'))
display(semillas_rnn.pivot(index='Modelo', columns='Semilla', values='RMSE').loc[list(arquitecturas)]
        .rename(columns=lambda s: f'Semilla {s}').round(4))

ensamble_semillas = {f'{nombre} (promedio de {len(SEMILLAS)} semillas)':
                     pd.concat(list(semillas_rnn.loc[semillas_rnn['Modelo'] == nombre, 'pred']), axis=1).mean(axis=1)
                     for nombre in arquitecturas}
print('Ensamble de semillas:')
display(tabla_metricas(ensamble_semillas).round(4))

plt.figure(figsize=(12, 4))
for i, nombre in enumerate(arquitecturas, start=1):
    valores = semillas_rnn.loc[semillas_rnn['Modelo'] == nombre, 'RMSE']
    plt.scatter([i] * len(valores), valores, color='goldenrod', s=50, zorder=3)
    plt.plot([i - 0.2, i + 0.2], [valores.mean()] * 2, color='black', linewidth=2)
plt.xticks(range(1, len(arquitecturas) + 1), list(arquitecturas))
plt.axhline(rmse_sarimax, color='crimson', linestyle='--', linewidth=1, label=f'{nombre_sarimax} ({rmse_sarimax:.2f})')
plt.axhline(comparacion_final.loc['CatBoost', 'RMSE'], color='steelblue', linestyle=':', linewidth=1,
            label=f"CatBoost ({comparacion_final.loc['CatBoost', 'RMSE']:.2f})")
plt.title(f'RMSE de test según la semilla (puntos = semillas, línea = promedio; hasta {EPOCAS} épocas con parada temprana)', fontweight='bold')
plt.ylabel('RMSE (puntos del índice)'); plt.grid(alpha=0.3); plt.legend(fontsize=8)
plt.tight_layout(); plt.show()

### 14.10. Comparación con todos los modelos

In [ ]:
nombres_rnn = {(conjunto, nombre): f'{nombre} ({conjunto.lower()})' for conjunto, nombre in pronosticos_rnn}
todos_dl = {**{nombres_rnn[c]: p for c, p in pronosticos_rnn.items()}, **ensamble_semillas}
tabla_dl = tabla_metricas(todos_dl)
tabla_dl.insert(0, 'Tipo', 'Deep Learning')
comparacion_dl = pd.concat([comparacion_final, tabla_dl]).sort_values('RMSE')
display(comparacion_dl.style.format('{:.4f}', subset=comparacion_dl.columns[1:])
        .apply(lambda fila: ['background-color: #fff2cc' if fila['Tipo'] == 'Deep Learning' else '' for _ in fila], axis=1)
        .highlight_min(subset=comparacion_dl.columns[1:], color='#c6efce')
        .set_caption('Deep Learning frente a SARIMAX, SARIMA, boosting, AutoML e ingenuos - test a 1 mes '
                     '(amarillo = Deep Learning, verde = mejor valor)'))

colores_tipo['Deep Learning'] = 'goldenrod'
fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(comparacion_dl.index, comparacion_dl['RMSE'], color=comparacion_dl['Tipo'].map(colores_tipo))
for i, valor in enumerate(comparacion_dl['RMSE']):
    ax.text(valor, i, f' {valor:.2f}', va='center', fontsize=8)
ax.invert_yaxis(); ax.grid(alpha=0.3, axis='x'); ax.set_xlabel('RMSE (puntos del índice)')
ax.set_title('RMSE de test a 1 mes (rojo = econometría, azul = boosting, naranja = AutoML,\namarillo = Deep Learning, gris = ingenuos)',
             fontweight='bold', fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
mejor_red = tabla_dl['RMSE'].idxmin()
rivales = {nombre_sarimax: pronosticos_1m[etiqueta_final], 'SARIMA': pronosticos_1m[etiqueta_sarima],
           'CatBoost': pronosticos_ml['CatBoost'], 'Ingenuo con último crecimiento': pronosticos_1m['Ingenuo con último crecimiento']}
dm_dl = []
for rival, pred_rival in rivales.items():
    dm, p = diebold_mariano(indice_test, todos_dl[mejor_red], pred_rival)
    dm_dl.append({'Comparación': f'{mejor_red} vs {rival}', 'Estadístico DM': dm, 'p-valor': p,
                  'Conclusión': 'Sin diferencia significativa' if p >= alpha else
                                'La red es significativamente mejor' if dm < 0 else f'{rival} es significativamente mejor'})
print('Mejor red (menor RMSE):', mejor_red)
display(pd.DataFrame(dm_dl).round(4))

### 14.11. Guardar y cargar el modelo para ponerlo en producción

In [ ]:
clave_mejor = min(pronosticos_rnn, key=lambda c: metricas_rnn.loc[c, 'RMSE'])
v = ventanas_rnn[clave_mejor[0]]
redes_rnn[clave_mejor].save('modelo_rnn_electricidad_agua.keras')
joblib.dump({'escalador_x': v['escalador_x'], 'escalador_y': v['escalador_y'], 'columnas': v['columnas'], 'ventana': VENTANA},
            'escaladores_rnn_electricidad_agua.joblib')
print('Modelo guardado:', nombres_rnn[clave_mejor], '-> modelo_rnn_electricidad_agua.keras (+ escaladores_rnn_electricidad_agua.joblib)')

modelo_cargado = load_model('modelo_rnn_electricidad_agua.keras')
modelo_cargado.summary()
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    diferencia = np.abs(modelo_cargado.predict(v['x_test'], verbose=0) - redes_rnn[clave_mejor].predict(v['x_test'], verbose=0)).max()
print(f'Diferencia máxima entre la red original y la cargada: {diferencia:.1e}')

## 15. Modelo a investigar: Transformer

### 15.1. Arquitectura: encoder con parches

In [ ]:
from keras.layers import MultiHeadAttention, LayerNormalization, Dropout, Reshape, Flatten
from keras.callbacks import EarlyStopping

D_MODELO = CONFIG['tr_d_modelo']
CABEZAS = CONFIG['tr_cabezas']
D_FF = CONFIG['tr_d_ff']
BLOQUES = CONFIG['tr_bloques']
TASA_DROPOUT = CONFIG['tr_dropout']
LARGOS_PARCHE = CONFIG['tr_largos_parche']
assert all(VENTANA % largo == 0 for largo in LARGOS_PARCHE)


class EmbeddingPosicional(keras.layers.Layer):
    def __init__(self, n_parches, d_modelo, **kwargs):
        super().__init__(**kwargs)
        self.n_parches, self.d_modelo = n_parches, d_modelo

    def build(self, input_shape):
        self.posicion = self.add_weight(name='posicion', shape=(self.n_parches, self.d_modelo),
                                        initializer='random_normal', trainable=True)

    def call(self, x):
        return x + self.posicion

    def get_config(self):
        return {**super().get_config(), 'n_parches': self.n_parches, 'd_modelo': self.d_modelo}


def crear_transformer(n_variables, largo_parche, ventana=VENTANA, nombre=None):
    n_parches = ventana // largo_parche
    entrada = Input(shape=(ventana, n_variables))
    x = Reshape((n_parches, largo_parche * n_variables))(entrada)
    x = Dense(D_MODELO)(x)
    x = EmbeddingPosicional(n_parches, D_MODELO)(x)
    for _ in range(BLOQUES):
        atencion, puntajes = MultiHeadAttention(num_heads=CABEZAS, key_dim=D_MODELO // CABEZAS,
                                                dropout=TASA_DROPOUT)(x, x, return_attention_scores=True)
        x = LayerNormalization(epsilon=1e-6)(x + Dropout(TASA_DROPOUT)(atencion))
        ff = Dense(D_FF, activation='gelu')(x)
        ff = Dense(D_MODELO)(Dropout(TASA_DROPOUT)(ff))
        x = LayerNormalization(epsilon=1e-6)(x + Dropout(TASA_DROPOUT)(ff))
    salida = Dense(1)(Dropout(TASA_DROPOUT)(Flatten()(x)))
    red = keras.Model(entrada, salida, name=nombre)
    red.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3), loss='mean_squared_error')
    return red, keras.Model(entrada, puntajes)


parametros_tr = pd.DataFrame({f'{conjunto} ({len(columnas)} variables)':
                              {f'Parche de {largo} mes(es), {VENTANA // largo} tokens':
                               crear_transformer(len(columnas), largo)[0].count_params() for largo in LARGOS_PARCHE}
                              for conjunto, columnas in conjuntos_rnn.items()})
print('Parámetros del Transformer:')
display(parametros_tr)
print('Parámetros de las redes recurrentes (sección 14):')
display(arquitectura_rnn.filter(like='Parámetros'))
crear_transformer(n_con, 3, nombre='Transformer_parches')[0].summary()

### 15.2. Entrenamiento con parada temprana

In [ ]:
LOTE = CONFIG['tr_lote']
EPOCAS_MAX = CONFIG['tr_epocas_max']
PACIENCIA = CONFIG['tr_paciencia']
N_VAL = CONFIG['tr_n_val']


def entrenar_transformer(v, largo_parche, semilla=0):
    x, y, n_var = v['x_train'], v['y_train'], len(v['columnas'])
    set_random_seed(semilla)
    red, _ = crear_transformer(n_var, largo_parche)
    parada = EarlyStopping(monitor='val_loss', patience=PACIENCIA, restore_best_weights=True)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        historia = red.fit(x[:-N_VAL], y[:-N_VAL], validation_data=(x[-N_VAL:], y[-N_VAL:]),
                           epochs=EPOCAS_MAX, batch_size=LOTE, callbacks=[parada], verbose=0).history
    mejor_epoca = int(np.argmin(historia['val_loss'])) + 1

    set_random_seed(semilla)
    red, red_atencion = crear_transformer(n_var, largo_parche)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        red.fit(x, y, epochs=mejor_epoca, batch_size=LOTE, verbose=0)
    return red, red_atencion, mejor_epoca, historia


def pronosticar_transformer(red, v, filtro=None):
    x = v['x_test'] if filtro is None else v['x_test'][filtro]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return v['escalador_y'].inverse_transform(red.predict(x, verbose=0))[:, 0]


v_ejemplo = ventanas_rnn['Sin exógenas']
print(f"Ventanas de train: {len(v_ejemplo['y_train'])} | etapa 1: {len(v_ejemplo['y_train']) - N_VAL} para entrenar y "
      f"{N_VAL} para validar ({v_ejemplo['fechas_train'][-N_VAL]:%Y-%m} a {v_ejemplo['fechas_train'][-1]:%Y-%m})")

### 15.3. Elección del largo del parche con la validación 2017-2019

In [ ]:
corte_val, fin_val = pd.Timestamp('2017-01-01'), pd.Timestamp('2019-12-01')

filas = []
for conjunto, columnas in conjuntos_rnn.items():
    v_val = crear_ventanas(columnas, VENTANA, corte=corte_val)
    evaluar = v_val['fechas_test'] <= fin_val
    fechas_val = v_val['fechas_test'][evaluar]
    for largo in LARGOS_PARCHE:
        inicio = time.time()
        red, _, epocas, _ = entrenar_transformer(v_val, largo, semilla=0)
        indice_val = indice.shift(1)[fechas_val].values * np.exp(pronosticar_transformer(red, v_val, evaluar))
        filas.append({'Variables': conjunto, 'Largo del parche (meses)': largo, 'Tokens': VENTANA // largo,
                      'Épocas elegidas': epocas,
                      'RMSE validación 2017-2019': np.sqrt(mean_squared_error(indice[fechas_val], indice_val)),
                      'Tiempo (s)': time.time() - inicio})
        print(f"{conjunto:<13} parche {largo} | {epocas:3d} épocas | RMSE {filas[-1]['RMSE validación 2017-2019']:.3f} "
              f"| {filas[-1]['Tiempo (s)']:.0f} s")
validacion_tr = pd.DataFrame(filas)

ingenuo_val = (indice.shift(12) * indice.shift(1) / indice.shift(13))[fechas_val]
print(f'\nReferencia: ingenuo con último crecimiento en 2017-2019, RMSE '
      f'{np.sqrt(mean_squared_error(indice[fechas_val], ingenuo_val)):.3f}')
display(validacion_tr.pivot(index='Largo del parche (meses)', columns='Variables', values='RMSE validación 2017-2019')
        .style.format('{:.3f}').highlight_min(color='#c6efce', axis=0)
        .set_caption('RMSE del índice a 1 mes en la validación 2017-2019 (verde = mejor parche de cada conjunto)'))

mejor_por_conjunto = (validacion_tr.loc[validacion_tr.groupby('Variables')['RMSE validación 2017-2019'].idxmin()]
                      .set_index('Variables'))
largo_elegido = mejor_por_conjunto['Largo del parche (meses)'].astype(int).to_dict()
conjunto_tr = mejor_por_conjunto['RMSE validación 2017-2019'].idxmin()
print('Largo del parche elegido:', largo_elegido, '| variables del Transformer principal:', conjunto_tr)

### 15.4. Entrenamiento final con varias semillas

In [ ]:
SEMILLAS_TR = range(CONFIG['tr_semillas'])
pronosticos_tr, historial_tr, epocas_tr, redes_tr, atencion_tr = {}, {}, {}, {}, {}
for conjunto, v_tr in ventanas_rnn.items():
    for semilla in SEMILLAS_TR:
        inicio = time.time()
        red, red_att, epocas, historia = entrenar_transformer(v_tr, largo_elegido[conjunto], semilla)
        clave = (conjunto, semilla)
        pronosticos_tr[clave] = a_indice_ml(pronosticar_transformer(red, v_tr))
        redes_tr[clave], atencion_tr[clave], epocas_tr[clave], historial_tr[clave] = red, red_att, epocas, historia
        print(f'{conjunto:<13} parche {largo_elegido[conjunto]} | semilla {semilla} | {epocas:3d} épocas | '
              f'{time.time() - inicio:5.1f} s en {dispositivo}')

nombres_tr = {conjunto: f'Transformer ({conjunto.lower()})' for conjunto in ventanas_rnn}
todos_tr = {**{nombres_tr[c]: pronosticos_tr[(c, 0)] for c in ventanas_rnn},
            **{f'{nombres_tr[c]} (promedio de {len(SEMILLAS_TR)} semillas)':
               pd.concat([pronosticos_tr[(c, s)] for s in SEMILLAS_TR], axis=1).mean(axis=1) for c in ventanas_rnn}}
metricas_tr = tabla_metricas(todos_tr)
display(metricas_tr.style.format('{:.4f}').highlight_min(color='#c6efce')
        .set_caption('Transformer - test a 1 mes, métricas sobre el índice (verde = mejor valor)'))

semillas_tr = pd.DataFrame([{'Variables': c, 'Semilla': s, 'Épocas': epocas_tr[(c, s)],
                             'RMSE': np.sqrt(mean_squared_error(indice_test, pronosticos_tr[(c, s)]))}
                            for c in ventanas_rnn for s in SEMILLAS_TR])
resumen_semillas_tr = (semillas_tr.pivot(index='Variables', columns='Semilla', values='RMSE')
                       .rename(columns=lambda s: f'RMSE semilla {s}'))
resumen_semillas_tr['RMSE promedio'] = semillas_tr.groupby('Variables')['RMSE'].mean()
resumen_semillas_tr['Desv. estándar'] = semillas_tr.groupby('Variables')['RMSE'].std()
resumen_semillas_tr['Épocas por semilla'] = semillas_tr.groupby('Variables')['Épocas'].apply(lambda e: ', '.join(map(str, e)))
print('Variabilidad entre semillas:')
display(resumen_semillas_tr.round(4))

mejor_red = tabla_dl['RMSE'].idxmin()
referencia_tr = pd.concat([comparacion_final.loc[[nombre_sarimax, 'SARIMA', 'CatBoost', 'Ingenuo con último crecimiento']],
                           tabla_dl.loc[[mejor_red]]])[['MSE', 'RMSE', 'MAE', 'MAPE (%)', 'RMSE var. % (pp)', 'IEC']]
print('Referencia (secciones 12 y 14):')
display(referencia_tr.round(4))

### 15.5. Curvas de aprendizaje y pronósticos en el test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for ax, conjunto in zip(axes, ventanas_rnn):
    historia = historial_tr[(conjunto, 0)]
    ax.plot(range(1, len(historia['loss']) + 1), historia['loss'], label='Entrenamiento')
    ax.plot(range(1, len(historia['val_loss']) + 1), historia['val_loss'], label=f'Validación interna ({N_VAL} meses)')
    ax.axvline(epocas_tr[(conjunto, 0)], color='red', linestyle=':', label=f'Época elegida ({epocas_tr[(conjunto, 0)]})')
    ax.set_title(f'Transformer {conjunto.lower()} (parche de {largo_elegido[conjunto]} meses)', fontweight='bold')
    ax.set_xlabel('Época'); ax.set_yscale('log'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
axes[0].set_ylabel('MSE del Log_Return escalado')
plt.tight_layout(); guardar_figura('ts_15_transformer_curvas.png'); plt.show()

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(indice['2019-01-01':inicio_test - pd.DateOffset(months=1)], color='gray', linewidth=1.2, label='Train')
ax.plot(indice_test, color='black', linewidth=1.6, label='Real (test)')
for nombre, pred in todos_tr.items():
    ax.plot(pred, linestyle='--' if 'promedio' in nombre else ':', linewidth=1.1, label=nombre)
ax.plot(todos_dl[mejor_red], color='goldenrod', linewidth=1.1, label=f'Mejor red recurrente: {mejor_red}')
ax.plot(pronosticos_1m[etiqueta_final], color='crimson', linewidth=1.3, label=nombre_sarimax)
ax.axvspan(test.index[0], test.index[-1], color='lightgray', alpha=0.3)
ax.set_title('Índice PBI Electricidad y Agua: pronóstico a 1 mes - Transformer', fontweight='bold')
ax.set_ylabel('Índice (2007 = 100)'); ax.grid(alpha=0.3); ax.legend(loc='upper left', fontsize=8, ncol=2)
plt.tight_layout(); guardar_figura('ts_15_transformer_pronostico.png'); plt.show()

valid_tr = pd.concat({'Real': indice_test, **todos_tr, nombre_sarimax: pronosticos_1m[etiqueta_final]}, axis=1)
valid_tr.index = valid_tr.index.strftime('%Y-%m')
print('Real vs pronosticado, últimos 12 meses del test:')
display(valid_tr.tail(12).round(2))

### 15.6. Matriz de atención

In [ ]:
largo = largo_elegido[conjunto_tr]
n_tokens = VENTANA // largo
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    puntajes = np.asarray(atencion_tr[(conjunto_tr, 0)].predict(ventanas_rnn[conjunto_tr]['x_test'], verbose=0))
matriz_atencion = puntajes.mean(axis=(0, 1))
atencion_recibida = matriz_atencion.mean(axis=0)

def meses_del_parche(i):
    return VENTANA - i * largo, VENTANA - (i + 1) * largo + 1

etiquetas_tr = [f't−{meses_del_parche(i)[0]}' if largo == 1 else f't−{meses_del_parche(i)[0]}…t−{meses_del_parche(i)[1]}'
                for i in range(n_tokens)]
tiene_estacional = np.array([any(meses_del_parche(i)[1] <= r <= meses_del_parche(i)[0] for r in (12, 24))
                             for i in range(n_tokens)])

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5), gridspec_kw={'width_ratios': [1, 1.3]})
imagen = axes[0].imshow(matriz_atencion, cmap='viridis')
axes[0].set_xticks(range(n_tokens), etiquetas_tr, rotation=90, fontsize=7)
axes[0].set_yticks(range(n_tokens), etiquetas_tr, fontsize=7)
axes[0].set_xlabel('Parche consultado (clave)'); axes[0].set_ylabel('Parche que consulta')
axes[0].set_title('Matriz de atención promedio (último bloque)', fontweight='bold')
fig.colorbar(imagen, ax=axes[0], fraction=0.046)
axes[1].bar(range(n_tokens), atencion_recibida, color=np.where(tiene_estacional, 'crimson', 'steelblue'))
axes[1].axhline(1 / n_tokens, color='black', linestyle='--', linewidth=0.8, label='Atención uniforme')
axes[1].set_xticks(range(n_tokens), etiquetas_tr, rotation=90, fontsize=7)
axes[1].set_title('Atención recibida por parche (rojo = contiene t−12 o t−24)', fontweight='bold')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3, axis='y')
plt.suptitle(f'Transformer {conjunto_tr.lower()}, parche de {largo} mes(es)', fontweight='bold')
plt.tight_layout(); guardar_figura('ts_15_transformer_atencion.png'); plt.show()

orden_atencion = pd.Series(atencion_recibida, index=etiquetas_tr).sort_values(ascending=False)
print('Parches con más atención:', ', '.join(f'{e} ({a:.3f})' for e, a in orden_atencion.head(3).items()))
print(f'Atención promedio de los parches con t−12 o t−24: {atencion_recibida[tiene_estacional].mean():.3f} | '
      f'resto: {atencion_recibida[~tiene_estacional].mean():.3f} | uniforme: {1 / n_tokens:.3f}')

### 15.7. Diebold–Mariano

In [ ]:
mejor_tr = metricas_tr['RMSE'].idxmin()
rivales_tr = {nombre_sarimax: pronosticos_1m[etiqueta_final], 'SARIMA': pronosticos_1m[etiqueta_sarima],
              f'Mejor red recurrente: {mejor_red}': todos_dl[mejor_red], 'CatBoost': pronosticos_ml['CatBoost'],
              'Ingenuo con último crecimiento': pronosticos_1m['Ingenuo con último crecimiento']}
dm_tr = []
for rival, pred_rival in rivales_tr.items():
    dm, p_valor = diebold_mariano(indice_test, todos_tr[mejor_tr], pred_rival)
    dm_tr.append({'Comparación': f'{mejor_tr} vs {rival}', 'Estadístico DM': dm, 'p-valor': p_valor,
                  'Conclusión': 'Sin diferencia significativa' if p_valor >= alpha else
                                'El Transformer es significativamente mejor' if dm < 0 else f'{rival} es significativamente mejor'})
print('Mejor Transformer (menor RMSE):', mejor_tr)
display(pd.DataFrame(dm_tr).round(4))

rmse_tr = metricas_tr.loc[mejor_tr, 'RMSE']
for nombre_ref, rmse_ref in [(nombre_sarimax, comparacion_final.loc[nombre_sarimax, 'RMSE']),
                             (f'la mejor red recurrente ({mejor_red})', tabla_dl.loc[mejor_red, 'RMSE']),
                             ('CatBoost', comparacion_final.loc['CatBoost', 'RMSE']),
                             ('el ingenuo con último crecimiento', comparacion_final.loc['Ingenuo con último crecimiento', 'RMSE'])]:
    print(f'- Frente a {nombre_ref} (RMSE {rmse_ref:.2f}): RMSE del Transformer {rmse_tr:.2f} '
          f'({100 * (rmse_tr / rmse_ref - 1):+.1f} %)')

### 15.8. Transformers preentrenados: TimesFM-3, PatchTST-FM-r2 y Toto 2.0

In [ ]:
import torch
from huggingface_hub.utils import disable_progress_bars
from huggingface_hub.utils import logging as registro_hf

disable_progress_bars()
registro_hf.set_verbosity_error()
os.environ.setdefault('HF_HUB_DISABLE_SYMLINKS_WARNING', '1')
os.environ.setdefault('TSFM_PYTHON_LOGGING_LEVEL', 'WARNING')
logging.getLogger('httpx').setLevel(logging.WARNING)
logging.getLogger('torch.utils.flop_counter').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', module='gluonts')
TAMANOS_TOTO = CONFIG['toto_tamanos']
dispositivo_fm = 'cuda' if torch.cuda.is_available() else 'cpu'
serie_fm = indice.astype('float32')
exog_fm = X[cols_exog].bfill().astype('float32')


def contextos(fechas):
    return [serie_fm[serie_fm.index < f].to_numpy() for f in fechas]


def n_parametros(red):
    return sum(p.numel() for p in red.parameters())


def cargar_timesfm():
    instalar('timesfm3', 'timesfm[torch]>=3.0.2')
    from timesfm3 import TimesFM3Forecaster
    red = TimesFM3Forecaster.from_pretrained('google/timesfm-3.0-pytorch', device=dispositivo_fm, per_core_batch_size=32)

    def predecir(fechas, con_exogenas=False):
        cov = [exog_fm[exog_fm.index <= f].to_numpy().T for f in fechas] if con_exogenas else None
        salidas = red.predict_batch(contextos(fechas), horizon=1, past_future_covariates=cov)
        return pd.Series([float(np.ravel(s.forecast)[0]) for s in salidas], index=fechas)

    return red, n_parametros(red.model), {'TimesFM-3': predecir,
                                          f'TimesFM-3 + {etiqueta_exog}': lambda fechas: predecir(fechas, True)}


def cargar_patchtst_fm():
    for modulo, paquete in [('deprecated', 'deprecated'), ('einops', 'einops'), ('datasets', 'datasets'), ('yaml', 'pyyaml')]:
        instalar(modulo, paquete)
    instalar('tsfm_public', '--no-deps', 'granite-tsfm>=0.3.9')
    from tsfm_public import PatchTSTFMForPrediction
    red = PatchTSTFMForPrediction.from_pretrained('ibm-granite/granite-timeseries-patchtst-fm-r2').to(dispositivo_fm).eval()

    def predecir(fechas):
        entradas = [torch.tensor(c, device=dispositivo_fm) for c in contextos(fechas)]
        with torch.no_grad():
            salida = red(past_values=entradas, prediction_length=1, quantile_levels=[0.5], return_dict=True)
        return pd.Series([float(q.float().reshape(-1)[0]) for q in salida.quantile_outputs], index=fechas)

    return red, n_parametros(red), {'PatchTST-FM-r2': predecir}


def cargar_toto(tamano):
    instalar('toto2', 'toto-2>=2.0.0')
    from toto2 import Toto2Model
    red = Toto2Model.from_pretrained(f'Datadog/Toto-2.0-{tamano}', map_location=dispositivo_fm).to(dispositivo_fm).eval()
    parche, mediana = red.config.patch_size, red.output_head.knots.index(0.5)

    def predecir(fechas):
        ctx = contextos(fechas)
        largo = parche * int(np.ceil(max(map(len, ctx)) / parche))
        valores = torch.zeros(len(ctx), 1, largo, device=dispositivo_fm)
        observado = torch.zeros(len(ctx), 1, largo, dtype=torch.bool, device=dispositivo_fm)
        for i, c in enumerate(ctx):
            valores[i, 0, largo - len(c):] = torch.as_tensor(c, device=dispositivo_fm)
            observado[i, 0, largo - len(c):] = True
        cuantiles = red.forecast({'target': valores, 'target_mask': observado,
                                  'series_ids': torch.zeros(len(ctx), 1, dtype=torch.long, device=dispositivo_fm)}, horizon=1)
        return pd.Series(cuantiles[mediana, :, 0, 0].float().cpu().numpy(), index=fechas)

    return red, n_parametros(red), {f'Toto-2.0-{tamano}': predecir}


fichas_fm = pd.DataFrame([
    {'Modelo': 'TimesFM-3', 'Empresa': 'Google', 'Publicado': 'ago-2026', 'Parámetros': '~330 M',
     '¿Usa exógenas?': 'Sí (covariables)', 'Licencia de los pesos': 'TimesFM Non-Commercial v1.0 (uso académico permitido)'},
    {'Modelo': 'PatchTST-FM-r2', 'Empresa': 'IBM', 'Publicado': 'sep-2026', 'Parámetros': '~385 M',
     '¿Usa exógenas?': 'No', 'Licencia de los pesos': 'Apache 2.0 / OpenMDW 1.0'},
    {'Modelo': 'Toto 2.0', 'Empresa': 'Datadog', 'Publicado': 'may-2026', 'Parámetros': '4 M a 2,5 B',
     '¿Usa exógenas?': 'No (anunciado para una versión futura)', 'Licencia de los pesos': 'Apache 2.0'},
]).set_index('Modelo')
print('Dispositivo de los modelos preentrenados:', dispositivo_fm)
display(fichas_fm)

### 15.9. Validación 2017-2019 y elección del modelo preentrenado

In [ ]:
fechas_val_fm = serie.index[(serie.index >= pd.Timestamp('2017-01-01')) & (serie.index <= pd.Timestamp('2019-12-01'))]
cargadores_fm = {'TimesFM-3': cargar_timesfm, 'PatchTST-FM-r2': cargar_patchtst_fm,
                 **{f'Toto-2.0-{t}': (lambda t=t: cargar_toto(t)) for t in TAMANOS_TOTO}}

val_fm, pronosticos_fm, carga_fm = {}, {}, []
for familia_fm, cargar in cargadores_fm.items():
    inicio_fm = time.time()
    try:
        red_fm, n_par, candidatos_fm = cargar()
        for nombre_fm, predecir in candidatos_fm.items():
            val_fm[nombre_fm] = predecir(fechas_val_fm)
            pronosticos_fm[nombre_fm] = predecir(test.index)
        carga_fm.append({'Modelo': familia_fm, 'Parámetros (millones)': n_par / 1e6,
                         'Tiempo (s)': time.time() - inicio_fm, 'Estado': 'OK'})
    except Exception as error:
        carga_fm.append({'Modelo': familia_fm, 'Estado': f'No disponible → {type(error).__name__}: {str(error)[:160]}'})
    finally:
        red_fm = candidatos_fm = predecir = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    print(f"{familia_fm:<16} {carga_fm[-1]['Estado']}")
carga_fm = pd.DataFrame(carga_fm).set_index('Modelo')
display(carga_fm.round(1))

real_val_fm = indice[fechas_val_fm]
referencias_val_fm = {'Ingenuo estacional': indice.shift(12)[fechas_val_fm],
                      'Ingenuo con último crecimiento': (indice.shift(12) * indice.shift(1) / indice.shift(13))[fechas_val_fm]}


def fila_val_fm(pred):
    fila = {'RMSE': np.sqrt(mean_squared_error(real_val_fm, pred)), 'MAE': mean_absolute_error(real_val_fm, pred),
            'MAPE (%)': 100 * np.mean(np.abs((real_val_fm - pred) / real_val_fm))}
    fila.update({f'RMSE {anio}': np.sqrt(mean_squared_error(real_val_fm.loc[str(anio)], pred.loc[str(anio)]))
                 for anio in (2017, 2018, 2019)})
    return fila


tabla_val_fm = pd.DataFrame({n: fila_val_fm(p) for n, p in {**val_fm, **referencias_val_fm}.items()}).T
tabla_val_fm.insert(0, 'Tipo', ['Preentrenado (zero-shot)' if n in val_fm else 'Referencia' for n in tabla_val_fm.index])
tabla_val_fm.loc['Transformer desde cero (15.3)', ['Tipo', 'RMSE']] = ['Referencia', validacion_tr['RMSE validación 2017-2019'].min()]
tabla_val_fm = tabla_val_fm.sort_values('RMSE')
columnas_val_fm = ['RMSE', 'MAE', 'MAPE (%)', 'RMSE 2017', 'RMSE 2018', 'RMSE 2019']
tabla_val_fm[columnas_val_fm] = tabla_val_fm[columnas_val_fm].astype(float)
display(tabla_val_fm.style.format('{:.3f}', subset=columnas_val_fm, na_rep='—')
        .highlight_min(subset=columnas_val_fm, color='#c6efce')
        .set_caption('Validación 2017-2019: índice pronosticado a 1 mes (verde = mejor valor de cada columna)'))

elegido_fm = tabla_val_fm.loc[list(val_fm), 'RMSE'].idxmin() if val_fm else None
if elegido_fm is None:
    print('Ningún modelo preentrenado pudo cargarse: revise la columna "Estado" de la tabla anterior.')
else:
    rmse_fm = tabla_val_fm.loc[elegido_fm, 'RMSE']
    print(f'Transformer preentrenado elegido: {elegido_fm} (RMSE de validación {rmse_fm:.3f})')
    for referencia in ['Ingenuo con último crecimiento', 'Ingenuo estacional', 'Transformer desde cero (15.3)']:
        rmse_ref = tabla_val_fm.loc[referencia, 'RMSE']
        print(f'- Frente a {referencia} (RMSE {rmse_ref:.3f}): {100 * (rmse_fm / rmse_ref - 1):+.1f} %')

### 15.10. Modelo preentrenado elegido en el test

In [ ]:
if elegido_fm is None:
    print('Ningún Transformer preentrenado pudo cargarse (ver 15.9): se omiten 15.10 y 15.11.')
else:
    nombre_elegido_fm = f'{elegido_fm} (elegido)'
    referencias_fm = {nombre_sarimax: pronosticos_1m[etiqueta_final], 'SARIMA': pronosticos_1m[etiqueta_sarima],
                      f'Transformer desde cero: {mejor_tr}': todos_tr[mejor_tr],
                      'Ingenuo con último crecimiento': pronosticos_1m['Ingenuo con último crecimiento'],
                      'Ingenuo estacional': pronosticos_1m['Ingenuo estacional']}
    tabla_fm_test = tabla_metricas({**{nombre_elegido_fm if n == elegido_fm else n: p for n, p in pronosticos_fm.items()},
                                    **referencias_fm}).sort_values('RMSE')
    tabla_fm_test.insert(0, 'Rol', ['Elegido en la validación' if n == nombre_elegido_fm else 'Referencia'
                                    if n in referencias_fm else 'Candidato no elegido' for n in tabla_fm_test.index])
    tabla_fm_test.insert(1, 'RMSE validación', [tabla_val_fm['RMSE'].get(n.replace(' (elegido)', ''), np.nan)
                                                for n in tabla_fm_test.index])
    columnas_fm = ['RMSE validación', 'MSE', 'RMSE', 'MAE', 'MAPE (%)', 'RMSE var. % (pp)', 'IEC']
    display(tabla_fm_test.style.format('{:.4f}', subset=columnas_fm, na_rep='—')
            .highlight_min(subset=columnas_fm, color='#c6efce')
            .apply(lambda fila: ['background-color: #e8e0f5' if fila['Rol'] == 'Elegido en la validación' else ''
                                 for _ in fila], axis=1)
            .set_caption('Transformers preentrenados en el test a 1 mes (morado = elegido sin mirar el test)'))

    if len(pronosticos_fm) >= 3:
        rho, p_rho = stats.spearmanr([tabla_val_fm.loc[n, 'RMSE'] for n in pronosticos_fm],
                                     [np.sqrt(mean_squared_error(indice_test, pronosticos_fm[n])) for n in pronosticos_fm])
        print(f'Correlación de Spearman entre el RMSE de validación y el de test de los {len(pronosticos_fm)} candidatos: '
              f'{rho:.2f} (p-valor {p_rho:.3f})')

    dm_fm = []
    for rival, pred_rival in referencias_fm.items():
        dm, p_valor = diebold_mariano(indice_test, pronosticos_fm[elegido_fm], pred_rival)
        dm_fm.append({'Comparación (1 mes)': f'{elegido_fm} vs {rival}', 'Estadístico DM': dm, 'p-valor': p_valor,
                      'Conclusión': 'Sin diferencia significativa' if p_valor >= alpha else
                                    f'{elegido_fm} es significativamente mejor' if dm < 0 else
                                    f'{rival} es significativamente mejor'})
    display(pd.DataFrame(dm_fm).round(4))

### 15.11. Gráficos de los modelos preentrenados

In [ ]:
if elegido_fm is not None:
    fig, axes = plt.subplots(1, 2, figsize=(17, 5.5), gridspec_kw={'width_ratios': [1, 1.7]})
    rmse_val_fm = tabla_val_fm['RMSE']
    axes[0].barh(rmse_val_fm.index, rmse_val_fm.values,
                 color=['indigo' if n == elegido_fm else 'mediumpurple' if n in val_fm else 'gray' for n in rmse_val_fm.index])
    for i, valor in enumerate(rmse_val_fm.values):
        axes[0].text(valor, i, f' {valor:.2f}', va='center', fontsize=8)
    axes[0].invert_yaxis(); axes[0].grid(alpha=0.3, axis='x')
    axes[0].set_title('RMSE a 1 mes en la validación 2017-2019\n(índigo = elegido, gris = referencias)', fontweight='bold')

    axes[1].plot(indice['2021-01-01':], color='black', linewidth=1.6, label='Real')
    axes[1].plot(pronosticos_fm[elegido_fm], color='indigo', linewidth=1.3, label=f'{elegido_fm} (preentrenado, zero-shot)')
    axes[1].plot(todos_tr[mejor_tr], color='mediumpurple', linestyle='--', linewidth=1.1,
                 label=f'Transformer desde cero: {mejor_tr}')
    axes[1].plot(pronosticos_1m[etiqueta_final], color='crimson', linestyle=':', linewidth=1.3, label=nombre_sarimax)
    axes[1].axvspan(test.index[0], test.index[-1], color='lightgray', alpha=0.3)
    axes[1].set_title('Pronóstico a 1 mes en el test', fontweight='bold')
    axes[1].set_ylabel('Índice (2007 = 100)'); axes[1].grid(alpha=0.3); axes[1].legend(loc='upper left', fontsize=8)
    plt.tight_layout(); guardar_figura('ts_15_preentrenados.png'); plt.show()

## 16. Comparación de todos los modelos del objetivo

### 16.1. Indicadores adicionales: MASE y U de Theil

In [ ]:
def nombre_corto(etiqueta):
    if ' - ' not in etiqueta:
        return etiqueta
    familia_m, orden_m = etiqueta.split(' - ', 1)
    return orden_m if familia_m in ('AR', 'MA', 'ARMA', 'ARIMA', 'SARIMA') else f"SARIMAX{orden_m}{familia_m.replace('SARIMAX', '')}"


escala_mase = (indice[train.index] - indice[train.index].shift(12)).abs().mean()
previo_test = indice.shift(1)[test.index]

def u_theil(pred):
    pred = pred.reindex(indice_test.index)
    return np.sqrt((((pred - indice_test) / previo_test) ** 2).sum() / (((indice_test - previo_test) / previo_test) ** 2).sum())

columnas_final = ['MSE', 'RMSE', 'MAE', 'MAPE (%)', 'MASE', 'U de Theil', 'RMSE var. % (pp)', 'IEC']

def tabla_completa(pronos):
    tabla = tabla_metricas(pronos)
    tabla['MASE'] = tabla['MAE'] / escala_mase
    tabla['U de Theil'] = [u_theil(pred) for pred in pronos.values()]
    return tabla[columnas_final]

print(f'Denominador del MASE (MAE del ingenuo estacional en train): {escala_mase:.3f} puntos del índice')

### 16.2. Tabla de los modelos del objetivo

In [ ]:
etiqueta_familia = {f: f'{f} - {mejores.loc[f, "Modelo"]}' for f in ['AR', 'MA', 'ARMA', 'ARIMA', 'SARIMA']}
param_boost = {p['Modelo']: f"{p['Árboles']} árboles, profundidad {p['Profundidad']}, tasa {p['Tasa de aprendizaje']}"
               for p in mejores_param}
try:
    lider_automl = automl_model_best.model_id.split('_AutoML')[0]
except Exception:
    lider_automl = 'líder del leaderboard'
conjunto_rnn_obj = 'Sin exógenas'

objetivo = [(f, 'Econometría', nombre_corto(etiqueta_familia[f]), pronosticos_1m[etiqueta_familia[f]])
            for f in ['AR', 'MA', 'ARMA', 'ARIMA', 'SARIMA']]
objetivo += [('SARIMAX (propuesto)', 'Econometría', nombre_corto(etiqueta_final), pronosticos_1m[etiqueta_final])]
objetivo += [(n, 'Prophet', f"changepoint {params_prophet['changepoint_prior_scale']}, "
                             f"estacionalidad {params_prophet['seasonality_prior_scale']}, reestimado cada mes",
              pronosticos_prophet[n]) for n in regresores_prophet]
objetivo += [(m, 'Boosting', param_boost[m], pronosticos_ml[m]) for m in ['XGBoost', 'LightGBM', 'CatBoost']]
objetivo += [('AutoML (H2O)', 'AutoML', lider_automl, pronosticos_ml['AutoML (H2O)'])]
objetivo += [(m, 'Deep Learning', f'{conjunto_rnn_obj.lower()}, {EPOCAS} épocas, semilla 0', pronosticos_rnn[(conjunto_rnn_obj, m)])
             for m in ['LSTM', 'GRU', 'LSTM bidireccional', 'GRU bidireccional']]
objetivo += [('Transformer', 'Transformer',
              f'desde cero, {conjunto_tr.lower()}, parche de {largo_elegido[conjunto_tr]} mes(es), semilla 0',
              pronosticos_tr[(conjunto_tr, 0)])]
if elegido_fm is not None:
    objetivo += [(f'{elegido_fm} (preentrenado)', 'Transformer preentrenado',
                  'zero-shot (sin reentrenar), elegido en la validación 2017-2019', pronosticos_fm[elegido_fm])]
objetivo += [(m, 'Referencia', '—', pronosticos_1m[m]) for m in ['Ingenuo con último crecimiento', 'Ingenuo estacional']]

p_ljung = resumen_metricas[('Ajuste (train)', 'p-Ljung-Box(24)')]
etiquetas_econ = {**etiqueta_familia, 'SARIMAX (propuesto)': etiqueta_final}
tabla_objetivo = tabla_completa({m: pred for m, _, _, pred in objetivo})
tabla_objetivo.insert(0, 'Tipo', [t for _, t, _, _ in objetivo])
tabla_objetivo.insert(1, 'Especificación', [e for _, _, e, _ in objetivo])
tabla_objetivo.insert(2, '¿Pasa Ljung–Box?', [('Sí' if p_ljung[etiquetas_econ[m]] >= alpha else 'No')
                                             if m in etiquetas_econ else '—' for m in tabla_objetivo.index])
tabla_objetivo = tabla_objetivo.sort_values('RMSE')
tabla_objetivo.insert(0, 'Ranking', range(1, len(tabla_objetivo) + 1))
display(tabla_objetivo.style.format('{:.4f}', subset=columnas_final)
        .highlight_min(subset=columnas_final, color='#c6efce')
        .set_caption('Modelos del objetivo - test a 1 mes, métricas sobre el índice (verde = mejor valor de cada columna)'))
tabla_objetivo.to_csv('comparacion_modelos_objetivo.csv', encoding='utf-8-sig')

### 16.3. Gráficos de la comparación

In [ ]:
colores_tipo['Transformer'], colores_tipo['Prophet'] = 'mediumpurple', 'teal'
colores_tipo['Transformer preentrenado'] = 'indigo'
fig, axes = plt.subplots(1, 3, figsize=(18, 6.5), sharey=True)
for ax, columna in zip(axes, ['RMSE', 'MASE', 'U de Theil']):
    ax.barh(tabla_objetivo.index, tabla_objetivo[columna], color=tabla_objetivo['Tipo'].map(colores_tipo))
    for i, valor in enumerate(tabla_objetivo[columna]):
        ax.text(valor, i, f' {valor:.2f}', va='center', fontsize=8)
    if columna != 'RMSE':
        ax.axvline(1, color='black', linestyle='--', linewidth=0.8)
    ax.set_title(columna, fontweight='bold'); ax.grid(alpha=0.3, axis='x')
axes[0].invert_yaxis()
plt.suptitle('Test a 1 mes (rojo = econometría, azul = boosting, naranja = AutoML, amarillo = Deep Learning, '
             'morado = Transformer, índigo = Transformer preentrenado, verde azulado = Prophet, gris = ingenuos)', fontweight='bold', fontsize=10)
plt.tight_layout(); guardar_figura('ts_16_comparacion_final.png'); plt.show()

mejor_por_tipo = tabla_objetivo[tabla_objetivo['Tipo'] != 'Referencia'].groupby('Tipo', sort=False)['RMSE'].idxmin()
pred_objetivo = {m: pred for m, _, _, pred in objetivo}
fig, ax = plt.subplots(figsize=(15, 5.5))
ax.plot(indice['2021-01-01':], color='black', linewidth=1.6, label='Real')
for tipo, modelo_tipo in mejor_por_tipo.items():
    ax.plot(pred_objetivo[modelo_tipo], color=colores_tipo[tipo], linewidth=1.2,
            linestyle='-' if tipo == 'Econometría' else '--', label=f'{tipo}: {modelo_tipo}')
ax.axvspan(test.index[0], test.index[-1], color='lightgray', alpha=0.3)
ax.set_title('Pronóstico a 1 mes en el test: mejor modelo de cada tipo', fontweight='bold')
ax.set_ylabel('Índice (2007 = 100)'); ax.grid(alpha=0.3); ax.legend(loc='upper left', fontsize=8)
plt.tight_layout(); guardar_figura('ts_16_mejor_por_tipo.png'); plt.show()

### 16.4. Diebold–Mariano: SARIMAX frente a cada modelo

In [ ]:
dm_final = []
for modelo_obj, tipo, _, pred in objetivo:
    if modelo_obj == 'SARIMAX (propuesto)':
        continue
    dm, p_valor = diebold_mariano(indice_test, pronosticos_1m[etiqueta_final], pred)
    dm_final.append({'Modelo': modelo_obj, 'Tipo': tipo, 'RMSE': tabla_objetivo.loc[modelo_obj, 'RMSE'],
                     'Estadístico DM (SARIMAX vs modelo)': dm, 'p-valor': p_valor,
                     'Conclusión': 'Sin diferencia significativa' if p_valor >= alpha else
                                   'El SARIMAX es significativamente mejor' if dm < 0 else
                                   'El modelo es significativamente mejor que el SARIMAX'})
dm_final = pd.DataFrame(dm_final).set_index('Modelo').sort_values('RMSE')
display(dm_final.style.format({'RMSE': '{:.4f}', 'Estadístico DM (SARIMAX vs modelo)': '{:.3f}', 'p-valor': '{:.4f}'})
        .set_caption('Diebold–Mariano (estadístico negativo = el SARIMAX tiene menor error cuadrático)'))

### 16.5. Tabla consolidada: todos los modelos

In [ ]:
econometricos = [e for e in pronosticos_1m if e in ajustes]
referencias = [e for e in pronosticos_1m if e not in ajustes]
pred_por_nombre = {**{nombre_corto(e): pronosticos_1m[e] for e in econometricos}, **pronosticos_prophet, **pronosticos_ml,
                   **todos_dl, **todos_tr, **pronosticos_fm, **{e: pronosticos_1m[e] for e in referencias}}
tipo_por_nombre = {**{nombre_corto(e): 'Econometría' for e in econometricos}, **dict.fromkeys(pronosticos_prophet, 'Prophet'),
                   **{m: 'AutoML' if 'AutoML' in m else 'Boosting' for m in pronosticos_ml},
                   **dict.fromkeys(todos_dl, 'Deep Learning'), **dict.fromkeys(todos_tr, 'Transformer'),
                   **dict.fromkeys(pronosticos_fm, 'Transformer preentrenado'),
                   **{e: 'Referencia' if 'Ingenuo' in e else 'Econometría' for e in referencias}}

ajuste_train = resumen_metricas['Ajuste (train)'].dropna(how='all').rename(index=nombre_corto)
consolidada = ajuste_train.join(tabla_completa(pred_por_nombre), how='right')
consolidada.insert(0, 'Tipo', consolidada.index.map(tipo_por_nombre))
consolidada.insert(4, '¿Pasa Ljung–Box?', np.where(consolidada['p-Ljung-Box(24)'].isna(), '—',
                                                    np.where(consolidada['p-Ljung-Box(24)'] >= alpha, 'Sí', 'No')))
consolidada = consolidada.sort_values('RMSE')
consolidada.insert(0, 'Ranking', range(1, len(consolidada) + 1))
print(f'Modelos y variantes evaluados: {len(consolidada)}')
print(consolidada['Tipo'].value_counts().to_string())

color_fila = {'Prophet': '#d9efed', 'Deep Learning': '#fff2cc', 'Transformer': '#e8e0f5', 'Transformer preentrenado': '#d9d2e9'}
display(consolidada.style
        .format('{:.4f}', subset=['AIC', 'BIC', 'p-Ljung-Box(24)'] + columnas_final, na_rep='—')
        .apply(lambda fila: [f"background-color: {color_fila[fila['Tipo']]}" if fila['Tipo'] in color_fila else ''
                             for _ in fila], axis=1)
        .highlight_min(subset=columnas_final + ['AIC', 'BIC'], color='#c6efce')
        .set_caption('Tabla consolidada: todos los modelos - test a 1 mes, métricas sobre el índice '
                     '(verde azulado = Prophet, amarillo = Deep Learning, morado = Transformer, '
                     'lila oscuro = Transformer preentrenado, verde = mejor valor)'))
consolidada.to_csv('comparacion_todos_los_modelos.csv', encoding='utf-8-sig')

### 16.6. Resumen de la comparación

In [ ]:
mejor_rmse, mejor_iec = tabla_objetivo['RMSE'].idxmin(), tabla_objetivo['IEC'].idxmin()
modelos_eval = tabla_objetivo[tabla_objetivo['Tipo'] != 'Referencia']
print(f'1. Mejor modelo por RMSE: {mejor_rmse} (RMSE {tabla_objetivo.loc[mejor_rmse, "RMSE"]:.2f}; '
      f'MAPE {tabla_objetivo.loc[mejor_rmse, "MAPE (%)"]:.2f} %; MASE {tabla_objetivo.loc[mejor_rmse, "MASE"]:.2f}).')
print(f'2. Mejor modelo por IEC: {mejor_iec} (IEC {tabla_objetivo.loc[mejor_iec, "IEC"]:.4f}).')
print('3. Mejor modelo de cada tipo:')
for tipo, modelo_tipo in mejor_por_tipo.items():
    fila = tabla_objetivo.loc[modelo_tipo]
    print(f'   - {tipo}: {modelo_tipo} | puesto {fila["Ranking"]} de {len(tabla_objetivo)} | RMSE {fila["RMSE"]:.2f} | '
          f'MAPE {fila["MAPE (%)"]:.2f} % | MASE {fila["MASE"]:.2f} | U {fila["U de Theil"]:.2f}')
print(f'4. Modelos con MASE < 1: {(modelos_eval["MASE"] < 1).sum()} de {len(modelos_eval)} | '
      f'con U de Theil < 1: {(modelos_eval["U de Theil"] < 1).sum()} de {len(modelos_eval)}.')
fila_tr = tabla_objetivo.loc['Transformer']
print(f'5. Transformer: puesto {fila_tr["Ranking"]} de {len(tabla_objetivo)} (RMSE {fila_tr["RMSE"]:.2f}); '
      f'Diebold–Mariano frente al SARIMAX: p-valor {dm_final.loc["Transformer", "p-valor"]:.4f} '
      f'({dm_final.loc["Transformer", "Conclusión"].lower()}).')
peores = dm_final.index[dm_final['Conclusión'] == 'El SARIMAX es significativamente mejor'].tolist()
mejores_que_sarimax = dm_final.index[dm_final['Conclusión'] == 'El modelo es significativamente mejor que el SARIMAX'].tolist()
print(f'6. El SARIMAX es significativamente mejor que: {", ".join(peores) if peores else "ninguno"}.')
print(f'   Significativamente mejores que el SARIMAX: {", ".join(mejores_que_sarimax) if mejores_que_sarimax else "ninguno"}.')
if elegido_fm is not None:
    nombre_fm_obj = f'{elegido_fm} (preentrenado)'
    fila_fm = tabla_objetivo.loc[nombre_fm_obj]
    print(f'7. Transformer preentrenado ({elegido_fm}, zero-shot, elegido en 2017-2019): puesto {fila_fm["Ranking"]} de '
          f'{len(tabla_objetivo)} (RMSE {fila_fm["RMSE"]:.2f}; el Transformer desde cero tiene RMSE {fila_tr["RMSE"]:.2f}); '
          f'Diebold–Mariano frente al SARIMAX: p-valor {dm_final.loc[nombre_fm_obj, "p-valor"]:.4f} '
          f'({dm_final.loc[nombre_fm_obj, "Conclusión"].lower()}).')

### 16.7. Pronóstico de los 5 mejores modelos

In [ ]:
mejores_5 = consolidada[consolidada['Tipo'] != 'Referencia'].head(5)
color_top = dict(zip(mejores_5.index, ['crimson', 'tab:blue', 'tab:orange', 'tab:green', 'tab:purple']))
horizonte = {**{nombre_corto(e): (pronosticos[e], bandas[e]) for e in econometricos},
             **{n: (pronosticos_prophet_h[n], bandas_prophet[n]) for n in pronosticos_prophet_h}}
con_horizonte = [m for m in mejores_5.index if m in horizonte] or [next(m for m in consolidada.index if m in horizonte)]

fig, axes = plt.subplots(2, 1, figsize=(14, 9))
for ax in axes:
    ax.plot(indice['2018-01-01':], color='black', linewidth=1.4, label='Real')
for m in con_horizonte:
    axes[0].plot(horizonte[m][0], color=color_top.get(m, 'crimson'), linestyle='-' if m == con_horizonte[0] else '--', label=m)
banda = horizonte[con_horizonte[0]][1]
axes[0].fill_between(test.index, banda.iloc[:, 0], banda.iloc[:, 1], color=color_top.get(con_horizonte[0], 'crimson'),
                     alpha=0.12, label=f'IC 95 % ({con_horizonte[0]})')
axes[0].set_title('Pronóstico de todo el test desde dic-2021 (horizonte 1 a 55 meses)', fontweight='bold')
for posicion, (m, fila) in enumerate(mejores_5.iterrows(), start=1):
    axes[1].plot(pred_por_nombre[m], color=color_top[m], linestyle='-' if posicion == 1 else '--',
                 label=f"{posicion}. {m} ({fila['Tipo']}, RMSE {fila['RMSE']:.2f})")
axes[1].set_title('5 mejores modelos: pronóstico a 1 mes en el test', fontweight='bold')
for ax in axes:
    ax.axvspan(test.index[0], test.index[-1], color='lightgray', alpha=0.4)
    ax.grid(alpha=0.3); ax.legend(loc='upper left', fontsize=8)
plt.tight_layout(); guardar_figura('ts_16_mejores_5_modelos.png'); plt.show()